# Peningkatan Kualitas Citra pada Domain Spasial
### *Spatial Domain Image Enhancement* dengan Berbagai *Spatial Filter*

**Mata Kuliah:** Pengolahan Citra Digital · Program Studi Teknik Informatika, Universitas Halu Oleo

---

## Tujuan Pembelajaran

Setelah menyelesaikan notebook ini, mahasiswa mampu:

1. Menjelaskan mekanisme operasi ketetanggaan (*neighborhood operation*), konvolusi, dan korelasi pada domain spasial.
2. Mengimplementasikan filter linear (rerata, Gaussian, Laplacian, gradien) dan non-linear (median, min/maks, *midpoint*, *alpha-trimmed*, kontraharmonik, bilateral, NLM).
3. **Memilih filter yang tepat berdasarkan karakteristik degradasi**, bukan berdasarkan kebiasaan.
4. Mengukur hasil peningkatan secara kuantitatif (PSNR, SSIM, dan **CER hasil OCR**), bukan hanya secara visual.

## Studi Kasus Utama

Seluruh bab menggunakan satu objek yang sama: **hasil pindai (scan) dokumen ijazah Universitas Halu Oleo**, dengan
degradasi berbeda-beda pada setiap bab. Setiap filter dipasangkan dengan satu kasus degradasi nyata yang memang
menjadi alasan filter tersebut diciptakan, lalu diuji dampaknya pada tugas hilir (*downstream task*):
**OCR dengan `pytesseract`** dan **deteksi tanda tangan dengan OpenCV**.

| Bab | Filter | Kasus degradasi | Tugas hilir |
|---|---|---|---|
| 4 | Rerata / *box* | Derau Gaussian ringan hasil pindai | PSNR/SSIM |
| 5 | Gaussian berbobot | Foto ijazah dari kamera HP, cahaya rendah | PSNR/SSIM |
| 6 | **Median** | *Salt & pepper* dari sensor pindai rusak | **CER OCR** |
| 7 | Min & Maks | Guratan teks pecah / bintik *pepper* | Ketebalan guratan |
| 8 | *Midpoint*, *alpha-trimmed*, kontraharmonik | Derau campuran | PSNR |
| 9 | **Median adaptif** | Impuls kerapatan tinggi (35%) | PSNR |
| 10 | Laplacian | Teks & garis tabel yang pudar | Kontras tepi |
| 11 | *Unsharp masking* / *high-boost* | Pindai tidak fokus (*defocus*) | **CER OCR** |
| 12 | Roberts, Prewitt, **Sobel**, Scharr | Dokumen miring + lokasi tanda tangan | Deskew & deteksi TTD |
| 13 | **Bilateral** | Derau pada foto pas 3×4 di ijazah | PSNR + profil tepi |
| 14 | *Non-local means* | Derau Gaussian kuat | PSNR |
| 15 | Filter statistik lokal & CLAHE | Iluminasi pindai tidak rata | **CER OCR** |
| 16 | Pipeline gabungan | Semua degradasi sekaligus | **CER OCR** |

> **Catatan lingkup:** notebook ini murni domain spasial. Transformasi Fourier, *wavelet*, dan CNN **tidak** dibahas.
> Semua citra dibangkitkan secara sintetis di dalam notebook, sehingga tidak ada berkas eksternal yang perlu diunduh.

---
## 1. Persiapan Lingkungan

Pustaka yang dibutuhkan: `numpy`, `opencv-python`, `matplotlib`, `scikit-image`, dan (opsional) `pytesseract`
beserta *binary* Tesseract.

```bash
pip install numpy opencv-python matplotlib scikit-image pytesseract
# Ubuntu/Debian: sudo apt install tesseract-ocr tesseract-ocr-ind
# Windows: unduh installer Tesseract, lalu set pytesseract.pytesseract.tesseract_cmd
```
Jika Tesseract tidak tersedia, notebook tetap berjalan penuh — bagian evaluasi OCR otomatis dilewati.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

plt.rcParams.update({"figure.dpi": 110, "image.cmap": "gray", "image.interpolation": "nearest",
                     "axes.titlesize": 9, "figure.max_open_warning": 0})
np.set_printoptions(precision=3, suppress=True, linewidth=110)

# --- ketersediaan OCR (opsional) ---
try:
    import pytesseract
    pytesseract.get_tesseract_version()
    OCR_ADA = True
except Exception as e:
    OCR_ADA = False
    print("Tesseract tidak tersedia -> bagian evaluasi OCR dilewati:", type(e).__name__)

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| OCR aktif:", OCR_ADA)

### 1.1 Fungsi utilitas

Tiga hal yang dipakai berulang: menampilkan citra berdampingan, menampilkan kernel sebagai matriks, dan
mengukur kualitas. **Penting:** semua `imshow` memakai `vmin=0, vmax=255` agar perbandingan antarpanel jujur —
tanpa itu, matplotlib menormalkan tiap panel secara terpisah dan filter yang buruk bisa tampak bagus.

In [ ]:
def tampil(items, cols=3, tinggi=3.4, judul=None):
    """Tampilkan daftar (citra, keterangan). Skala intensitas dikunci 0-255 agar adil."""
    n = len(items)
    cols = min(cols, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * tinggi))
    axes = np.atleast_1d(axes).ravel()
    for ax, (img, ket) in zip(axes, items):
        a = np.asarray(img)
        if a.dtype != np.uint8:                      # citra float (mis. peta gradien)
            ax.imshow(a, cmap="gray")
        else:
            ax.imshow(a, vmin=0, vmax=255)
        ax.set_title(ket)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    if judul:
        fig.suptitle(judul, fontsize=11, y=1.0)
    fig.tight_layout()
    plt.show()


def tampil_kernel(K, nama="Kernel", fmt="{:.4f}"):
    print(f"{nama}  (ukuran {K.shape[0]}x{K.shape[1]}, jumlah bobot = {K.sum():.4f})")
    for r in K:
        print("  [" + "  ".join(fmt.format(v) for v in r) + "]")


def metrik(ref, uji, label=""):
    """PSNR (dB, makin tinggi makin baik) dan SSIM (0-1, kemiripan struktur)."""
    p = psnr(ref, uji, data_range=255)
    s = ssim(ref, uji, data_range=255)
    if label:
        print(f"  {label:<34s} PSNR = {p:6.2f} dB   SSIM = {s:.4f}")
    return p, s


def bandingkan(ref, kandidat):
    """kandidat: list of (citra, nama). Cetak tabel metrik terurut PSNR."""
    hasil = [(nama, *metrik(ref, img)) for img, nama in kandidat]
    lebar = max(len(n) for n, _, _ in hasil) + 2
    print(f"{'Metode':<{lebar}}{'PSNR (dB)':>11}{'SSIM':>9}")
    print("-" * (lebar + 20))
    for nama, p, s in sorted(hasil, key=lambda t: -t[1]):
        print(f"{nama:<{lebar}}{p:>11.2f}{s:>9.4f}")
    return hasil

### 1.2 Metrik berbasis tugas hilir: CER hasil OCR

PSNR dan SSIM mengukur kedekatan dengan citra asli, **bukan** kegunaan citra untuk sistem yang memakainya.
Untuk verifikasi ijazah, yang penting adalah teksnya terbaca. Kita pakai **CER** (*Character Error Rate*):

$$\mathrm{CER} = \frac{S + D + I}{N} = \frac{\text{jarak Levenshtein(prediksi, acuan)}}{\text{panjang acuan}}$$

CER = 0 berarti sempurna; CER ≥ 1 berarti keluaran OCR sudah tidak berguna.

In [ ]:
def levenshtein(a, b):
    """Jarak edit klasik, implementasi DP O(len(a)*len(b)) dengan dua baris."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1,          # hapus
                           cur[j - 1] + 1,       # sisip
                           prev[j - 1] + (ca != cb)))  # substitusi
        prev = cur
    return prev[-1]


def normalisasi(t):
    """Rapikan spasi/kapital agar CER mengukur isi, bukan tata letak."""
    return " ".join(t.replace("|", "").split()).lower()


def baca_ocr(img, psm=6):
    if not OCR_ADA:
        return ""
    return pytesseract.image_to_string(img, config=f"--psm {psm}")


def cer(img, acuan, psm=6):
    """CER hasil OCR terhadap teks acuan. Mengembalikan (cer, teks_terbaca)."""
    if not OCR_ADA:
        return float("nan"), ""
    hyp = normalisasi(baca_ocr(img, psm))
    ref = normalisasi(acuan)
    return levenshtein(hyp, ref) / max(len(ref), 1), hyp

---
## 2. Membangkitkan Citra Uji: Pindaian Ijazah

Kita bangkitkan ijazah sintetis (data fiktif) agar tersedia **citra acuan bebas degradasi**. Tanpa acuan,
PSNR/SSIM tidak bisa dihitung dan mahasiswa hanya bisa menilai "kelihatannya lebih bagus" — yang menyesatkan.

Citra ini sengaja memuat empat jenis struktur dengan tuntutan filter yang berbeda:

| Struktur | Letak | Karakter | Tuntutan |
|---|---|---|---|
| Teks kecil (isian data) | tengah | detail frekuensi tinggi, guratan 1–2 piksel | jangan dikaburkan |
| Garis kop & tabel | atas | tepi lurus kontras tinggi | jangan bergerigi |
| Tanda tangan & stempel | kanan bawah | guratan kurvatur tinggi, kontras rendah (stempel) | jangan hilang |
| Foto pas 3×4 | kiri bawah | area halus + tepi tegas | halus tanpa merusak tepi |

Resolusi disetel lewat konstanta `SKALA` (baku 1,5 → 1350×1020 px, tinggi huruf isian ±22 px). Resolusi ini
penting: pada teks yang lebih kecil dari ±15 px, Tesseract gagal bahkan pada citra bersih, sehingga metrik CER
menjadi tidak informatif untuk membandingkan filter.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

RNG = np.random.default_rng(42)
SKALA = 1.5          # 1.0 = 900x680 px. Naikkan bila ingin resolusi lebih tinggi.

# Font: sesuaikan bila menjalankan di Windows/macOS.
KANDIDAT_SERIF  = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf", "/System/Library/Fonts/Times.ttc",
                   "times.ttf", "C:/Windows/Fonts/times.ttf"]
KANDIDAT_SERIFB = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf", "timesbd.ttf", "C:/Windows/Fonts/timesbd.ttf"]
KANDIDAT_SANS   = ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", "/System/Library/Fonts/Helvetica.ttc",
                   "arial.ttf", "C:/Windows/Fonts/arial.ttf"]

def _font(kandidat, size):
    for p in kandidat:
        try:
            return ImageFont.truetype(p, size)
        except Exception:
            continue
    return ImageFont.load_default()


def buat_ijazah(s=SKALA):
    """Ijazah sintetis grayscale uint8 pada skala s. Semua data bersifat fiktif.
    Tinggi huruf isian ~= 22 px pada s=1.5, setara pindaian 200-300 DPI."""
    w, h = int(900 * s), int(680 * s)
    S = lambda v: int(round(v * s))
    img = Image.new("L", (w, h), 246)
    d = ImageDraw.Draw(img)
    d.rectangle([S(12), S(12), w - S(12), h - S(12)], outline=90, width=S(3))
    d.rectangle([S(22), S(22), w - S(22), h - S(22)], outline=140, width=max(1, S(1)))

    def tengah(t, y, font, fill=25):
        bb = d.textbbox((0, 0), t, font=font)
        d.text(((w - (bb[2] - bb[0])) / 2, S(y)), t, font=font, fill=fill)

    # --- kop ---
    tengah("KEMENTERIAN PENDIDIKAN TINGGI, SAINS, DAN TEKNOLOGI", 42, _font(KANDIDAT_SANS, S(13)), 60)
    tengah("UNIVERSITAS HALU OLEO", 62, _font(KANDIDAT_SERIFB, S(26)))
    tengah("KENDARI - SULAWESI TENGGARA", 96, _font(KANDIDAT_SANS, S(12)), 70)
    d.line([S(180), S(118), w - S(180), S(118)], fill=90, width=S(2))
    tengah("I J A Z A H", 134, _font(KANDIDAT_SERIFB, S(22)))
    tengah("Nomor: 4471/UN29.1/PP/2026", 166, _font(KANDIDAT_SANS, S(12)), 70)

    # --- isian data (target OCR) ---
    body, label = _font(KANDIDAT_SERIF, S(15)), _font(KANDIDAT_SANS, S(13))
    baris = [("Nama", "La Ode Muhammad Fahri"),
             ("Tempat / Tanggal Lahir", "Baubau, 17 Agustus 2003"),
             ("Nomor Induk Mahasiswa", "E1E1 22 045"),
             ("Program Studi", "S1 Teknik Informatika"),
             ("Fakultas", "Teknik"),
             ("Tanggal Lulus", "12 Juni 2026")]
    for i, (k, v) in enumerate(baris):
        y = S(210 + i * 30)
        d.text((S(80), y), k, font=label, fill=80)
        d.text((S(300), y), ": " + v, font=body, fill=25)
    d.text((S(230), S(424)), "Gelar: Sarjana Teknik (S.T.)", font=body, fill=25)
    d.text((S(230), S(452)), "Predikat: Pujian (Cum Laude)", font=body, fill=25)

    # --- blok tanda tangan ---
    d.text((S(560), S(408)), "Kendari, 12 Juni 2026", font=label, fill=60)
    d.text((S(560), S(430)), "Rektor,", font=label, fill=60)
    ttd = [(S(575) + t, S(520) - 0.12 * t - S(26) * np.sin(t / (24 * s))
            - S(9) * np.sin(t / (8 * s)) - S(4) * np.sin(t / (3 * s))) for t in range(0, S(200), 2)]
    d.line(ttd, fill=20, width=S(3), joint="curve")
    d.line([S(565), S(556), S(800), S(556)], fill=90, width=max(1, S(1)))
    d.text((S(565), S(562)), "Prof. Dr. Andi Wahyudin, S.Si., M.Si.", font=_font(KANDIDAT_SANS, S(11)), fill=40)

    # --- stempel (kontras rendah, sengaja menimpa TTD seperti dokumen nyata) ---
    d.ellipse([S(496), S(452), S(606), S(562)], outline=150, width=S(3))
    d.ellipse([S(508), S(464), S(594), S(550)], outline=170, width=max(1, S(1)))
    d.text((S(520), S(498)), "UHO", font=_font(KANDIDAT_SERIFB, S(20)), fill=160)

    # --- foto pas 3x4 sintetis: area halus + tepi tegas ---
    fh, fw = S(150), S(112)
    foto = np.zeros((fh, fw), np.float64)
    yy, xx = np.mgrid[0:fh, 0:fw]
    cx, cy = fw / 2, fh * 0.48
    foto += 150 + 40 * np.cos((xx - cx) / (70 * s))                                  # gradasi latar studio
    foto[(((xx - cx) / (34 * s)) ** 2 + ((yy - cy) / (46 * s)) ** 2) < 1] = 205       # wajah
    foto[(((xx - cx) / (26 * s)) ** 2 + ((yy - fh * 0.27) / (22 * s)) ** 2) < 1] = 120  # rambut
    foto[int(fh * 0.87):, :] = 90                                                    # kerah
    img.paste(Image.fromarray(foto.clip(0, 255).astype(np.uint8)), (S(80), S(440)))
    d.rectangle([S(80), S(440), S(80) + fw, S(440) + fh], outline=60, width=S(2))

    a = np.asarray(img).astype(np.float64) + RNG.normal(0, 1.2, (h, w))   # tekstur kertas halus
    return a.clip(0, 255).astype(np.uint8)


IJAZAH = buat_ijazah()          # citra acuan (ground truth) untuk seluruh notebook
print("Citra acuan:", IJAZAH.shape, IJAZAH.dtype, "| rentang", IJAZAH.min(), "-", IJAZAH.max())
print("Skala:", SKALA, "-> tinggi huruf isian sekitar", int(round(15 * SKALA)), "px")
tampil([(IJAZAH, "IJAZAH — citra acuan bebas degradasi")], cols=1, tinggi=6.0)

### 2.1 Wilayah amatan (ROI) dan teks acuan OCR

Menilai filter pada seluruh halaman menyembunyikan efek lokal. Kita definisikan ROI tetap agar setiap bab
memeriksa bagian yang paling sensitif terhadap filter yang sedang dibahas.

In [ ]:
# ROI didefinisikan pada koordinat dasar (skala 1.0) lalu diskalakan otomatis
_ROI_DASAR = {
    "data":    (195, 400,  60, 845),   # blok isian -> target OCR
    "kop":     ( 30, 130, 150, 750),   # garis & huruf besar
    "ttd":     (395, 600, 450, 880),   # tanda tangan + stempel
    "foto":    (435, 595,  75, 196),   # foto pas 3x4
    "teks1":   (205, 245, 290, 620),   # satu baris teks, untuk zoom guratan
}
ROI = {k: tuple(int(round(v * SKALA)) for v in v4) for k, v4 in _ROI_DASAR.items()}

def potong(img, nama):
    y0, y1, x0, x1 = ROI[nama]
    return img[y0:y1, x0:x1]

# Teks acuan untuk perhitungan CER pada ROI "data"
TEKS_ACUAN = ("Nama : La Ode Muhammad Fahri "
              "Tempat / Tanggal Lahir : Baubau, 17 Agustus 2003 "
              "Nomor Induk Mahasiswa : E1E1 22 045 "
              "Program Studi : S1 Teknik Informatika "
              "Fakultas : Teknik "
              "Tanggal Lulus : 12 Juni 2026")

c0, teks0 = cer(potong(IJAZAH, "data"), TEKS_ACUAN)
print(f"CER pada citra acuan (batas bawah yang bisa dicapai) = {c0:.3f}")
print("Hasil OCR acuan:", teks0[:150], "...")

tampil([(potong(IJAZAH, "data"), "ROI data (target OCR)"),
        (potong(IJAZAH, "ttd"), "ROI tanda tangan + stempel"),
        (potong(IJAZAH, "foto"), "ROI foto pas 3x4")], cols=3, tinggi=3.0)

> **Diskusi.** CER pada citra acuan tidak nol. Tesseract sudah salah baca beberapa karakter bahkan pada citra
> sempurna. Angka ini adalah **batas bawah** (*floor*) — sebuah filter dinilai bagus bila mendorong CER
> mendekati nilai ini, bukan ke nol. Melaporkan "akurasi 100%" pada pipeline OCR nyata hampir selalu tanda
> kesalahan metodologi.

### 2.2 Model degradasi

Setiap fungsi di bawah memodelkan satu penyebab nyata pada alur pindai dokumen. Perhatikan bahwa **jenis
degradasi menentukan filter yang tepat** — inti dari seluruh notebook ini.

| Fungsi | Penyebab fisik | Model matematis |
|---|---|---|
| `derau_gaussian` | derau termal/elektronik sensor | $g = f + \eta,\ \eta \sim \mathcal{N}(0,\sigma^2)$ |
| `derau_impulse` | bit rusak, sensor mati, kompresi gagal | piksel diganti 0 atau 255 dengan peluang $p$ |
| `derau_seragam` | kuantisasi kasar | $g = f + \eta,\ \eta \sim U(-a,a)$ |
| `kabur_defokus` | fokus lensa/kaca pindai kotor | $g = f * h_{\text{Gauss}}$ |
| `iluminasi_tidak_rata` | lampu pindai lemah / dokumen tak rata | $g = f \cdot s(x,y)$ (multiplikatif) |
| `kertas_kusam` | kertas tua, kontras rendah | penyempitan rentang intensitas |

In [ ]:
def derau_gaussian(img, sigma=18, seed=0):
    r = np.random.default_rng(seed)
    return (img.astype(np.float64) + r.normal(0, sigma, img.shape)).clip(0, 255).astype(np.uint8)

def derau_impulse(img, p=0.08, rasio_salt=0.5, seed=0):
    r = np.random.default_rng(seed)
    out, m = img.copy(), r.random(img.shape)
    batas = p * rasio_salt
    out[m < batas] = 255                       # salt (bintik putih)
    out[(m >= batas) & (m < p)] = 0            # pepper (bintik hitam)
    return out

def derau_seragam(img, a=35, seed=0):
    r = np.random.default_rng(seed)
    return (img.astype(np.float64) + r.uniform(-a, a, img.shape)).clip(0, 255).astype(np.uint8)

def kabur_defokus(img, sigma=2.2):
    return cv2.GaussianBlur(img, (0, 0), sigma)

def iluminasi_tidak_rata(img, kuat=0.55):
    h, w = img.shape
    yy, xx = np.mgrid[0:h, 0:w]
    s = 1 - kuat * np.sqrt((xx / w - 0.15) ** 2 + (yy / h - 0.10) ** 2)
    return (img.astype(np.float64) * s).clip(0, 255).astype(np.uint8)

def kertas_kusam(img, lo=95, hi=205):
    f = img.astype(np.float64) / 255.0
    return (lo + f * (hi - lo)).clip(0, 255).astype(np.uint8)


contoh = [(IJAZAH, "acuan"),
          (derau_gaussian(IJAZAH, 18), "derau Gaussian sigma=18"),
          (derau_impulse(IJAZAH, 0.08), "impulse p=0.08"),
          (kabur_defokus(IJAZAH, 2.2), "defokus sigma=2.2"),
          (iluminasi_tidak_rata(IJAZAH), "iluminasi tidak rata"),
          (kertas_kusam(IJAZAH), "kertas kusam")]
tampil([(potong(i, "data"), k) for i, k in contoh], cols=3, tinggi=2.4,
       judul="Model degradasi pada ROI data")

---
## 3. Dasar Penyaringan Spasial

### 3.1 Definisi

Filter spasial bekerja **langsung pada nilai piksel** — tidak ada transformasi ke domain lain. Nilai keluaran
sebuah piksel ditentukan oleh nilai piksel di sekitarnya (ketetanggaan) yang tercakup oleh sebuah jendela
berukuran $m \times n$ (umumnya ganjil, $m = 2a+1$, $n = 2b+1$).

**Korelasi** — jendela digeser apa adanya:

$$g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\, f(x+s,\, y+t)$$

**Konvolusi** — jendela diputar 180° lebih dahulu:

$$g(x,y) = (w \ast f)(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\, f(x-s,\, y-t)$$

Untuk kernel simetris (rerata, Gaussian, Laplacian) keduanya identik. Untuk kernel **asimetris**
(Sobel, Prewitt) hasilnya berbeda tanda/arah. Ini sumber bug yang sangat sering terjadi:
`cv2.filter2D` sesungguhnya menghitung **korelasi**, bukan konvolusi. Untuk konvolusi sejati, kernel harus
dibalik sendiri dengan `cv2.flip(K, -1)`.

In [ ]:
def korelasi_manual(img, K, border=cv2.BORDER_REFLECT_101):
    """Implementasi eksplisit (lambat) untuk membuktikan rumus. Jangan dipakai di produksi."""
    K = np.asarray(K, np.float64)
    m, n = K.shape
    a, b = m // 2, n // 2
    p = cv2.copyMakeBorder(img, a, a, b, b, border).astype(np.float64)
    out = np.zeros(img.shape, np.float64)
    for s in range(-a, a + 1):
        for t in range(-b, b + 1):
            out += K[s + a, t + b] * p[a + s: a + s + img.shape[0], b + t: b + t + img.shape[1]]
    return out


# --- verifikasi terhadap OpenCV memakai kernel ASIMETRIS ---
K_asim = np.array([[0, 0, 0],
                   [1, 0, -1],
                   [0, 0, 0]], np.float64)
uji = IJAZAH[200:280, 300:420]

manual   = korelasi_manual(uji, K_asim)
cv_korel = cv2.filter2D(uji, cv2.CV_64F, K_asim, borderType=cv2.BORDER_REFLECT_101)
cv_konv  = cv2.filter2D(uji, cv2.CV_64F, cv2.flip(K_asim, -1), borderType=cv2.BORDER_REFLECT_101)

print("|manual - filter2D(K)|        max =", np.abs(manual - cv_korel).max(), " -> filter2D = KORELASI")
print("|manual - filter2D(flip(K))|  max =", np.abs(manual - cv_konv).max(), " -> berbeda (tanda terbalik)")
print("konvolusi = -korelasi di sini?", np.allclose(cv_konv, -cv_korel))

### 3.2 Penanganan tepi (*border handling*)

Pada piksel tepi, jendela menjulur ke luar citra. Pilihan strategi bukan sekadar detail teknis: pada dokumen
pindai, `BORDER_CONSTANT` (isi nol/hitam) menciptakan garis gelap palsu di pinggir yang nantinya terdeteksi
sebagai tepi oleh Sobel dan dianggap garis tabel. Untuk dokumen, gunakan `REFLECT_101` atau `REPLICATE`.

In [ ]:
K_rerata = np.ones((15, 15), np.float64) / 225.0
tepi = IJAZAH[:70, :180]                       # sudut kiri atas: ada bingkai
mode = [("BORDER_CONSTANT (0)", cv2.BORDER_CONSTANT), ("BORDER_REPLICATE", cv2.BORDER_REPLICATE),
        ("BORDER_REFLECT_101", cv2.BORDER_REFLECT_101), ("BORDER_WRAP", cv2.BORDER_WRAP)]
tampil([(tepi, "asli (sudut kiri atas)")] +
       [(cv2.filter2D(tepi, -1, K_rerata, borderType=m), f"rerata 15x15 · {nm}") for nm, m in mode],
       cols=5, tinggi=2.2, judul="Efek strategi tepi pada filter rerata 15x15")

### 3.3 Separabilitas: mengapa Gaussian murah

Kernel $w$ disebut *separable* bila $w = c\, r^{T}$, yakni hasil kali vektor kolom dan vektor baris. Konvolusi 2D
$m \times m$ berbiaya $O(m^2)$ operasi per piksel; bila dipisah menjadi dua konvolusi 1D biayanya turun menjadi
$O(2m)$. Untuk $m = 31$: 961 → 62 operasi, sekitar **15× lebih cepat**, dengan hasil identik.

In [ ]:
import time

k1d = cv2.getGaussianKernel(31, 5.0)                  # vektor kolom 31x1
K2d = k1d @ k1d.T                                     # kernel 2D 31x31
print("rank kernel 2D =", np.linalg.matrix_rank(K2d), "(rank 1 => separable)")

t = time.perf_counter(); a2 = cv2.filter2D(IJAZAH, cv2.CV_64F, K2d);            t2d = time.perf_counter() - t
t = time.perf_counter(); a1 = cv2.sepFilter2D(IJAZAH, cv2.CV_64F, k1d, k1d);    t1d = time.perf_counter() - t
print(f"2D penuh   : {t2d*1000:7.2f} ms")
print(f"dua kali 1D: {t1d*1000:7.2f} ms   (percepatan {t2d/max(t1d,1e-9):.1f}x)")
print("selisih hasil maksimum:", np.abs(a2 - a1).max())

---
## 4. Filter Rerata (*Mean / Box / Averaging Filter*)

### Teori

Semua bobot bernilai sama dan berjumlah 1, sehingga keluaran adalah rata-rata aritmetik ketetanggaan:

$$g(x,y) = \frac{1}{mn}\sum_{(s,t)\in S_{xy}} f(s,t), \qquad
w = \frac{1}{9}\begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}$$

Karena derau Gaussian bersifat aditif dan berrerata nol, merata-ratakan $N = mn$ piksel menekan simpangan bakunya
menjadi $\sigma/\sqrt{N}$ — **inilah dasar teoretis mengapa filter ini bekerja**. Konsekuensinya: sinyal
frekuensi tinggi (guratan huruf) juga tertekan. Filter ini adalah *low-pass*, dan ia tidak bisa membedakan
derau dari detail.

### Studi Kasus 4 — Derau elektronik sensor pindai

Ijazah dipindai dengan pemindai tua di ruang arsip; sensornya menghasilkan derau Gaussian kuat
($\sigma \approx 35$). Pertanyaan praktis: **ukuran kernel berapa yang optimal?**

In [ ]:
DERAU_G = derau_gaussian(IJAZAH, sigma=35, seed=7)
metrik(IJAZAH, DERAU_G, "citra terdegradasi (baseline)")

hasil = []
for k in [3, 5, 7, 9, 15]:
    hasil.append((cv2.blur(DERAU_G, (k, k)), f"rerata {k}x{k}"))

tampil([(potong(DERAU_G, "teks1"), "terdegradasi (zoom teks)")] +
       [(potong(h, "teks1"), n) for h, n in hasil], cols=3, tinggi=1.6,
       judul="Studi Kasus 4: pengaruh ukuran kernel rerata pada guratan huruf")

print()
_ = bandingkan(IJAZAH, [(DERAU_G, "tanpa filter")] + hasil)

In [ ]:
# Kurva PSNR DAN SSIM terhadap ukuran kernel -> keduanya tidak sepakat, dan itu informatif
ks = np.arange(3, 22, 2)
kp = [psnr(IJAZAH, cv2.blur(DERAU_G, (int(k), int(k))), data_range=255) for k in ks]
kss = [ssim(IJAZAH, cv2.blur(DERAU_G, (int(k), int(k))), data_range=255) for k in ks]
k_psnr, k_ssim = int(ks[int(np.argmax(kp))]), int(ks[int(np.argmax(kss))])

fig, ax1 = plt.subplots(figsize=(6.6, 3.4))
ax1.plot(ks, kp, "o-", c="steelblue", label="PSNR")
ax1.axhline(psnr(IJAZAH, DERAU_G, data_range=255), ls="--", c="steelblue", alpha=.5, label="PSNR tanpa filter")
ax1.set_xlabel("ukuran kernel (piksel)"); ax1.set_ylabel("PSNR (dB)", color="steelblue")
ax2 = ax1.twinx()
ax2.plot(ks, kss, "s-", c="darkorange", label="SSIM")
ax2.axhline(ssim(IJAZAH, DERAU_G, data_range=255), ls="--", c="darkorange", alpha=.5, label="SSIM tanpa filter")
ax2.set_ylabel("SSIM", color="darkorange")
ax1.axvline(k_psnr, ls=":", c="steelblue"); ax2.axvline(k_ssim, ls=":", c="darkorange")
fig.legend(loc="lower center", ncol=4, fontsize=7, bbox_to_anchor=(0.5, -0.06))
ax1.set_title(f"PSNR optimum di k={k_psnr}, SSIM optimum di k={k_ssim}")
ax1.grid(alpha=.3); fig.tight_layout(); plt.show()

print(f"Kernel optimum menurut PSNR : {k_psnr}x{k_psnr}")
print(f"Kernel optimum menurut SSIM : {k_ssim}x{k_ssim}")

> **Analisis — dua metrik, dua kesimpulan.** PSNR memilih kernel kecil, SSIM memilih kernel besar. Penyebabnya:
> PSNR adalah rerata galat kuadrat, sehingga **didominasi oleh galat besar di tepi huruf** yang berkontras tinggi;
> begitu kernel diperbesar, galat tepi meledak. SSIM menimbang keseragaman lokal, dan sebagian besar luas ijazah
> adalah **kertas kosong** yang memang menjadi jauh lebih bersih dengan kernel besar.
>
> Pelajaran praktisnya: pada citra dokumen, jangan menyetel ukuran kernel dengan satu metrik saja. Kernel rerata
> $\ge 5\times5$ mulai menyatukan guratan huruf yang berdekatan — secara visual jelas terlihat di panel zoom di
> atas, sekalipun SSIM masih naik. Filter rerata karenanya layak dipakai hanya sebagai **prapemrosesan ringan**
> ($3\times3$) atau sebagai **estimator latar belakang** dengan kernel besar (dipakai di Bab 15), bukan sebagai
> penghalus utama dokumen.

---
## 5. Filter Rerata Berbobot dan Gaussian

### Teori

Kelemahan filter rerata: piksel di sudut jendela diberi pengaruh yang sama dengan piksel pusat, padahal
korelasinya dengan piksel pusat lebih lemah. Filter berbobot memberi bobot menurun terhadap jarak. Bentuk
paling penting adalah Gaussian:

$$w(s,t) = \frac{1}{2\pi\sigma^{2}} e^{-\frac{s^{2}+t^{2}}{2\sigma^{2}}}$$

Tiga sifat yang membuatnya menjadi filter penghalus standar:

1. **Isotropik** — menghaluskan sama di semua arah, tidak menghasilkan artefak berbentuk kotak seperti *box filter*.
2. **Separable** — biaya $O(2m)$, lihat Bab 3.3.
3. **Tidak menghasilkan *ringing*** — transformasi Fouriernya juga Gaussian, tanpa *sidelobe*.

**Aturan praktis ukuran kernel:** $m \approx 2\lceil 3\sigma \rceil + 1$. Kernel yang terlalu kecil untuk $\sigma$
besar memotong ekor Gaussian dan berperilaku seperti *box filter*. Di OpenCV, `ksize=(0,0)` membuat ukuran
dihitung otomatis dari `sigma` — **inilah cara yang dianjurkan**.

### Studi Kasus 5 — Ijazah difoto dengan kamera HP pada cahaya rendah

Pemohon mengunggah foto ijazah, bukan hasil pindai. ISO tinggi menghasilkan derau Gaussian kuat ($\sigma = 25$).
Bandingkan *box* vs Gaussian pada ukuran kernel yang sama.

In [ ]:
k1d5 = cv2.getGaussianKernel(5, 1.0)
tampil_kernel(np.ones((5, 5)) / 25.0, "Box 5x5")
print()
tampil_kernel(k1d5 @ k1d5.T, "Gaussian 5x5 (sigma=1.0)")

In [ ]:
DERAU_HP = derau_gaussian(IJAZAH, sigma=25, seed=11)

pasangan = []
for k, s in [(5, 1.0), (7, 1.4), (9, 1.8)]:
    pasangan.append((cv2.blur(DERAU_HP, (k, k)), f"box {k}x{k}"))
    pasangan.append((cv2.GaussianBlur(DERAU_HP, (k, k), s), f"gauss {k}x{k} s={s}"))

tampil([(potong(p, "teks1"), n) for p, n in pasangan], cols=2, tinggi=1.6,
       judul="Studi Kasus 5: box vs Gaussian pada ukuran kernel identik")
print()
_ = bandingkan(IJAZAH, [(DERAU_HP, "tanpa filter")] + pasangan)

In [ ]:
# Pencarian sigma optimum -- praktik yang benar: kunci sigma, biarkan ksize otomatis
sigmas = np.arange(0.4, 3.01, 0.2)
kurva = [psnr(IJAZAH, cv2.GaussianBlur(DERAU_HP, (0, 0), s), data_range=255) for s in sigmas]
s_opt = float(sigmas[int(np.argmax(kurva))])

plt.figure(figsize=(6.2, 3.2))
plt.plot(sigmas, kurva, "o-")
plt.axvline(s_opt, ls=":", c="crimson", label=f"sigma optimum = {s_opt:.1f}")
plt.xlabel("sigma"); plt.ylabel("PSNR (dB)"); plt.title("Pemilihan sigma Gaussian (ksize otomatis)")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

best = cv2.GaussianBlur(DERAU_HP, (0, 0), s_opt)
tampil([(potong(DERAU_HP, "data"), f"terdegradasi (PSNR {psnr(IJAZAH, DERAU_HP, data_range=255):.2f} dB)"),
        (potong(best, "data"), f"Gaussian sigma={s_opt:.1f} (PSNR {psnr(IJAZAH, best, data_range=255):.2f} dB)")],
       cols=2, tinggi=2.6)

> **Analisis.** Pada ukuran kernel yang sama, Gaussian selalu unggul dari *box filter*: PSNR lebih tinggi **dan**
> guratan huruf lebih terjaga, karena bobot pusat yang dominan mempertahankan struktur lokal. Namun keduanya tetap
> filter linear *low-pass* — tepi ikut kabur. Untuk mempertahankan tepi, diperlukan filter non-linear
> (Bab 13: bilateral).

---
## 6. Filter Median

### Teori

Filter median adalah filter **statistik urutan** (*order-statistic*): jendela diurutkan, lalu diambil nilai
tengahnya.

$$g(x,y) = \operatorname{median}\{\, f(s,t) \mid (s,t)\in S_{xy} \,\}$$

Filter ini **non-linear**: $\operatorname{median}(f_1 + f_2) \ne \operatorname{median}(f_1) + \operatorname{median}(f_2)$,
sehingga tidak bisa dinyatakan sebagai kernel konvolusi dan tidak memiliki fungsi transfer.

Keunggulannya terhadap derau impuls bersifat struktural, bukan kebetulan. Nilai 0 dan 255 dari *salt & pepper*
selalu berada di **ujung** daftar terurut, sehingga tidak pernah terpilih sebagai median selama jumlah piksel
terkontaminasi dalam jendela kurang dari setengah. Untuk jendela $3\times3$ (9 piksel), median tetap benar sampai
4 piksel rusak. Filter rerata sebaliknya: satu piksel bernilai 255 langsung menggeser rerata sebesar $255/9 \approx 28$
tingkat keabuan, dan justru **menyebarkan** kontaminasi ke seluruh jendela.

### Studi Kasus 6 — Sensor pindai rusak (*salt & pepper* 8%), dinilai dengan CER OCR

Berkas ijazah dari kantor akademik memuat bintik hitam-putih akibat baris sensor yang rusak. Ini kasus paling
penting untuk pipeline verifikasi ijazah karena **OCR gagal total** pada citra seperti ini.

In [ ]:
IMP = derau_impulse(IJAZAH, p=0.08, seed=1)

kand = [(IMP, "tanpa filter"),
        (cv2.blur(IMP, (3, 3)), "rerata 3x3"),
        (cv2.GaussianBlur(IMP, (3, 3), 0.8), "Gaussian 3x3"),
        (cv2.medianBlur(IMP, 3), "median 3x3"),
        (cv2.medianBlur(IMP, 5), "median 5x5")]

tampil([(potong(k, "teks1"), n) for k, n in kand], cols=1, tinggi=1.5,
       judul="Studi Kasus 6: derau impuls -- filter linear vs median")
print()
_ = bandingkan(IJAZAH, kand)

In [ ]:
# Evaluasi berbasis tugas hilir: CER hasil OCR
print(f"{'Metode':<22}{'CER':>8}   Cuplikan hasil OCR")
print("-" * 100)
print(f"{'ACUAN (tanpa derau)':<22}{c0:>8.3f}   {teks0[:58]}")
for img, nama in kand:
    nilai, teks = cer(potong(img, "data"), TEKS_ACUAN)
    print(f"{nama:<22}{nilai:>8.3f}   {teks[:58]}")

> **Analisis.** Perhatikan pemisahan yang tegas antara dua keluarga filter:
>
> - **Filter linear (rerata/Gaussian) menaikkan PSNR tetapi tidak menyelamatkan keterbacaan.** Ini kasus paling
>   berbahaya dalam praktik: angka membaik, dokumen tetap tidak terbaca. Secara fisik, rerata mengubah satu
>   bintik menjadi tambalan kabur berukuran $3\times3$ — jumlah piksel yang tercemar justru **bertambah**
>   sembilan kali, hanya dengan amplitudo yang lebih kecil. PSNR menyukai perubahan itu (galat besar dipecah
>   menjadi banyak galat kecil); OCR tidak.
> - **Median 3×3 memulihkan keterbacaan** hampir ke tingkat citra acuan, karena bintik ekstrem tersingkir dari
>   posisi tengah daftar terurut.
> - **Median 5×5 mulai menghapus detail**: guratan huruf tipis yang lebarnya < 3 piksel bisa hilang sepenuhnya bila
>   mayoritas jendela adalah kertas putih. Untuk teks berukuran kecil, **median 3×3 hampir selalu pilihan yang tepat**.
>
> Aturan praktis untuk pipeline dokumen: bila histogram menunjukkan puncak terisolasi di 0 dan 255,
> gunakan median, **jangan pernah** filter rerata.

In [ ]:
# Bukti diagnostik: histogram memperlihatkan tanda tangan khas derau impuls
fig, ax = plt.subplots(1, 3, figsize=(12, 2.9))
for a, (img, nama) in zip(ax, [(IJAZAH, "acuan"), (IMP, "impuls 8%"), (cv2.medianBlur(IMP, 3), "median 3x3")]):
    a.hist(img.ravel(), bins=256, range=(0, 255), color="steelblue")
    a.set_yscale("log"); a.set_title(f"histogram: {nama}"); a.set_xlabel("intensitas")
plt.tight_layout(); plt.show()
print("Puncak terisolasi di 0 dan 255 pada panel tengah adalah tanda diagnostik derau impuls.")

---
## 7. Filter Minimum dan Maksimum

### Teori

Masih keluarga statistik urutan, tetapi mengambil ujung daftar, bukan tengahnya:

$$g_{\min}(x,y) = \min_{(s,t)\in S_{xy}} f(s,t), \qquad g_{\max}(x,y) = \max_{(s,t)\in S_{xy}} f(s,t)$$

Keduanya identik dengan **erosi** dan **dilasi** morfologi grayscale, sehingga di OpenCV cukup memakai
`cv2.erode` / `cv2.dilate`.

Pada dokumen dengan **tinta gelap di atas kertas terang**, efeknya spesifik dan mudah diingat:

| Filter | Efek pada nilai | Efek pada tinta gelap | Menghapus |
|---|---|---|---|
| Minimum (erosi) | mengambil yang paling gelap | guratan **menebal** | derau *salt* (bintik putih) |
| Maksimum (dilasi) | mengambil yang paling terang | guratan **menipis**/hilang | derau *pepper* (bintik hitam) |

Keduanya menggeser tingkat keabuan rata-rata (menggelapkan atau menerangkan citra) — ini harga yang harus dibayar
dan alasan mengapa keduanya jarang dipakai sebagai penghalus umum.

### Studi Kasus 7 — Guratan huruf pecah karena bintik *salt*

Ijazah lama difoto dengan pantulan cahaya (*glare*) yang memutihkan titik-titik kecil, sehingga guratan huruf
terpotong-potong dan OCR salah membaca. Ini kasus khas untuk filter **minimum**.

In [ ]:
SALT   = derau_impulse(IJAZAH, p=0.06, rasio_salt=1.0, seed=5)   # hanya bintik putih
PEPPER = derau_impulse(IJAZAH, p=0.06, rasio_salt=0.0, seed=5)   # hanya bintik hitam
se3 = np.ones((3, 3), np.uint8)

kasus_salt = [(SALT, "salt 6% (guratan pecah)"),
              (cv2.erode(SALT, se3), "MIN 3x3 (erosi)"),
              (cv2.dilate(SALT, se3), "MAX 3x3 (dilasi) - salah pilih"),
              (cv2.medianBlur(SALT, 3), "median 3x3 (pembanding)")]
tampil([(potong(k, "teks1"), n) for k, n in kasus_salt], cols=1, tinggi=1.5,
       judul="Studi Kasus 7a: derau salt -- filter MIN adalah pilihan yang tepat")
print("Derau SALT (bintik putih) -- metrik acuan:")
_ = bandingkan(IJAZAH, kasus_salt)
print(f"\nEvaluasi tugas hilir (CER acuan tanpa derau = {c0:.3f}):")
for img, nama in kasus_salt:
    print(f"  {nama:<34}CER = {cer(potong(img, 'data'), TEKS_ACUAN)[0]:.3f}")

In [ ]:
kasus_pepper = [(PEPPER, "pepper 6%"),
                (cv2.dilate(PEPPER, se3), "MAX 3x3 (dilasi)"),
                (cv2.erode(PEPPER, se3), "MIN 3x3 - salah pilih"),
                (cv2.medianBlur(PEPPER, 3), "median 3x3 (pembanding)")]
tampil([(potong(k, "teks1"), n) for k, n in kasus_pepper], cols=1, tinggi=1.5,
       judul="Studi Kasus 7b: derau pepper -- filter MAX adalah pilihan yang tepat")
print("Derau PEPPER (bintik hitam) -- metrik acuan:")
_ = bandingkan(IJAZAH, kasus_pepper)
print(f"\nEvaluasi tugas hilir (CER acuan tanpa derau = {c0:.3f}):")
for img, nama in kasus_pepper:
    print(f"  {nama:<34}CER = {cer(potong(img, 'data'), TEKS_ACUAN)[0]:.3f}")

In [ ]:
# Pengukuran ketebalan guratan: proporsi piksel tinta setelah binarisasi Otsu
def rasio_tinta(img):
    _, bw = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return bw.mean() / 255.0

roi_teks = potong(IJAZAH, "data")
print(f"{'Citra':<26}{'rasio tinta':>13}{'perubahan':>12}")
print("-" * 51)
dasar = rasio_tinta(roi_teks)
for img, nama in [(IJAZAH, "acuan"),
                  (cv2.erode(IJAZAH, se3), "MIN 3x3"),
                  (cv2.dilate(IJAZAH, se3), "MAX 3x3"),
                  (cv2.medianBlur(IJAZAH, 3), "median 3x3")]:
    r = rasio_tinta(potong(img, "data"))
    print(f"{nama:<26}{r:>13.4f}{(r/dasar - 1)*100:>11.1f}%")
print("\nMIN menebalkan tinta (rasio naik), MAX menipiskannya (rasio turun), median hampir netral.")

> **Analisis — dan sebuah peringatan tentang metrik.** Tiga hal terbaca dari angka di atas:
>
> 1. **Memilih arah yang salah bersifat katastrofik.** Filter maksimum pada derau *salt*, atau filter minimum
>    pada derau *pepper*, menghasilkan citra yang jauh lebih buruk daripada tidak memfilter sama sekali —
>    perhatikan PSNR yang bisa jatuh ke kisaran 4–17 dB dan CER yang melewati 1,0 (keluaran OCR tidak berguna).
> 2. **Pada kasus *salt*, PSNR dan CER bertolak belakang.** PSNR filter minimum jatuh sekitar 10 dB di bawah
>    citra berderau dan jauh di bawah median, sehingga menurut PSNR ia termasuk kandidat terburuk — tetapi
>    CER-nya justru **yang terbaik dari semuanya**. Penyebabnya: filter minimum menggelapkan seluruh halaman
>    dan menebalkan setiap guratan, sehingga galat kuadrat terhadap acuan melonjak — padahal justru penebalan
>    itulah yang menyambung kembali guratan yang terputus oleh bintik putih. Kalau keputusan diambil dari PSNR
>    saja, kesimpulannya salah.
> 3. **Pada kasus *pepper*, median lebih unggul dari filter maksimum** pada kedua metrik. Filter maksimum
>    memang menghapus bintik hitam, tetapi sekaligus menipiskan guratan (lihat tabel rasio tinta: −88%),
>    sehingga sebagian huruf ikut hilang.
>
> Kesimpulan praktis: median tetap pilihan baku untuk derau impuls. Nilai filter min/maks terletak pada
> **efek sampingnya yang terkendali** — menebalkan atau menipiskan guratan secara sengaja — dan pada perannya
> sebagai balok penyusun filter lain (*midpoint* pada Bab 8 dan median adaptif pada Bab 9 keduanya dibangun
> dari min dan maks).

---
## 8. Keluarga Filter Rerata dan Statistik Urutan Lainnya

Empat filter berikut jarang diajarkan tetapi sangat berguna ketika derau **tidak murni** satu jenis.

**Rerata geometrik** — menghaluskan setara filter rerata namun kehilangan detail lebih sedikit; sangat sensitif
terhadap piksel bernilai 0 (satu piksel hitam membuat hasil menjadi 0):

$$g(x,y) = \left[\prod_{(s,t)\in S_{xy}} f(s,t)\right]^{\frac{1}{mn}} = \exp\left(\frac{1}{mn}\sum \ln f(s,t)\right)$$

**Rerata kontraharmonik** — satu parameter $Q$ mengatur polaritas derau yang dihapus:

$$g(x,y) = \frac{\sum f(s,t)^{Q+1}}{\sum f(s,t)^{Q}}$$

$Q > 0$ menghapus *pepper*; $Q < 0$ menghapus *salt*; $Q = 0$ menjadi rerata aritmetik; $Q = -1$ menjadi rerata harmonik.

**Titik tengah (*midpoint*)** — rerata dari nilai ekstrem, terbaik untuk derau berdistribusi simetris
(Gaussian, seragam):

$$g(x,y) = \tfrac{1}{2}\left[\max_{S_{xy}} f + \min_{S_{xy}} f\right]$$

**Rerata terpangkas-alfa (*alpha-trimmed mean*)** — buang $d/2$ nilai terendah dan $d/2$ tertinggi, lalu rata-ratakan
sisanya. Filter ini adalah **jembatan** antara rerata ($d=0$) dan median ($d = mn-1$), dan menjadi pilihan terbaik
untuk derau campuran:

$$g(x,y) = \frac{1}{mn-d}\sum_{(s,t)\in S_{xy}^{\,\text{terpangkas}}} f(s,t)$$

### Studi Kasus 8 — Derau campuran: seragam ($a=30$) + impuls (4%)

Ijazah difoto dengan kamera murah (kuantisasi kasar → derau seragam) lalu dikirim lewat aplikasi pesan yang
merusak sebagian blok (→ derau impuls). Tidak ada filter tunggal dari bab sebelumnya yang ideal di sini.

In [ ]:
def rerata_geometrik(img, k=3):
    f = np.log(img.astype(np.float64) + 1.0)
    return (np.exp(cv2.boxFilter(f, -1, (k, k))) - 1.0).clip(0, 255).astype(np.uint8)

def rerata_kontraharmonik(img, k=3, Q=1.5):
    f = img.astype(np.float64) + 1e-3
    atas  = cv2.boxFilter(np.power(f, Q + 1), -1, (k, k))
    bawah = cv2.boxFilter(np.power(f, Q),     -1, (k, k))
    return (atas / (bawah + 1e-12)).clip(0, 255).astype(np.uint8)

def filter_midpoint(img, k=3):
    se = np.ones((k, k), np.uint8)
    return ((cv2.erode(img, se).astype(np.float64) + cv2.dilate(img, se).astype(np.float64)) / 2
            ).clip(0, 255).astype(np.uint8)

def rerata_terpangkas_alfa(img, k=3, d=4):
    """d harus genap dan < k*k. d=0 -> rerata aritmetik; d=k*k-1 -> median."""
    assert d % 2 == 0 and d < k * k, "d harus genap dan lebih kecil dari k*k"
    p = cv2.copyMakeBorder(img, k // 2, k // 2, k // 2, k // 2, cv2.BORDER_REFLECT_101)
    jend = sliding_window_view(p, (k, k)).reshape(img.shape[0], img.shape[1], k * k)
    urut = np.sort(jend, axis=2)
    inti = urut[:, :, d // 2: k * k - d // 2]          # buang d/2 terkecil dan d/2 terbesar
    return inti.mean(axis=2).clip(0, 255).astype(np.uint8)


CAMPUR = derau_impulse(derau_seragam(IJAZAH, a=30, seed=2), p=0.04, seed=3)

kand8 = [(CAMPUR, "tanpa filter"),
         (cv2.blur(CAMPUR, (3, 3)), "rerata aritmetik 3x3"),
         (rerata_geometrik(CAMPUR, 3), "rerata geometrik 3x3"),
         (rerata_kontraharmonik(CAMPUR, 3, +1.5), "kontraharmonik Q=+1.5"),
         (rerata_kontraharmonik(CAMPUR, 3, -1.5), "kontraharmonik Q=-1.5"),
         (filter_midpoint(CAMPUR, 3), "midpoint 3x3"),
         (cv2.medianBlur(CAMPUR, 3), "median 3x3"),
         (rerata_terpangkas_alfa(CAMPUR, 3, 4), "alpha-trimmed 3x3 (d=4)"),
         (rerata_terpangkas_alfa(CAMPUR, 5, 12), "alpha-trimmed 5x5 (d=12)")]

tampil([(potong(k, "teks1"), n) for k, n in kand8], cols=1, tinggi=1.45,
       judul="Studi Kasus 8: derau campuran (seragam + impuls)")
print()
_ = bandingkan(IJAZAH, kand8)

In [ ]:
# Pengaruh parameter d pada alpha-trimmed: dari rerata (d=0) menuju median (d=8)
plt.figure(figsize=(6.2, 3.2))
ds = [0, 2, 4, 6, 8]
nilai = []
for d in ds:
    h = rerata_terpangkas_alfa(CAMPUR, 3, d) if d < 9 else cv2.medianBlur(CAMPUR, 3)
    nilai.append(psnr(IJAZAH, h, data_range=255))
plt.plot(ds, nilai, "o-")
plt.xticks(ds, [f"d={d}" + ("\n(=rerata)" if d == 0 else "\n(≈median)" if d == 8 else "") for d in ds])
plt.ylabel("PSNR (dB)"); plt.title("Alpha-trimmed sebagai jembatan rerata <-> median (3x3)")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print("d terbaik =", ds[int(np.argmax(nilai))], "-> derau campuran butuh kompromi, bukan salah satu ekstrem.")

> **Analisis.** Empat pengamatan dari tabel:
>
> - **Kontraharmonik sangat bergantung pada tanda $Q$.** $Q=+1{,}5$ dan $Q=-1{,}5$ memberi hasil berlawanan, dan
>   tanda yang salah merusak citra lebih parah daripada tidak memfilter sama sekali.
> - ***Midpoint* dan rerata geometrik runtuh.** *Midpoint* justru **memakai** nilai ekstrem yang merupakan
>   deraunya; rerata geometrik hancur karena satu piksel bernilai 0 (bintik *pepper*) menarik seluruh hasil
>   perkalian ke nol. Keduanya baik untuk derau simetris tanpa impuls, dan hanya untuk itu.
> - **Median dan alpha-trimmed berada di kelas yang sama** pada derau campuran ini, dengan median sedikit
>   unggul pada PSNR dan alpha-trimmed $5\times5$ unggul jelas pada SSIM. Artinya komponen impuls-lah yang
>   mendominasi kerusakan di sini.
> - **Kurva $d$ adalah alat diagnosis.** Nilai $d$ terbaik memberi tahu jenis derau mana yang dominan:
>   $d$ mendekati 0 berarti derau aditif yang dominan, $d$ mendekati $mn-1$ berarti impuls yang dominan.
>   Pada percobaan ini optimumnya bergeser ke arah median — konsisten dengan pengamatan sebelumnya.

---
## 9. Filter Median Adaptif

### Teori

Median berukuran tetap gagal pada dua situasi: kerapatan derau tinggi (> ~20%, mayoritas jendela rusak sehingga
median pun terkontaminasi) dan kernel besar (detail ikut hilang). Median adaptif mengatasi keduanya dengan
**membesarkan jendela hanya jika perlu**, per piksel.

Notasi: $z_{\min}, z_{\max}, z_{\text{med}}$ = nilai minimum/maksimum/median dalam jendela $S_{xy}$;
$z_{xy}$ = nilai piksel pusat; $S_{\max}$ = ukuran jendela maksimum yang diizinkan.

**Tahap A** — apakah median itu sendiri sebuah impuls?
- $A_1 = z_{\text{med}} - z_{\min}$, $A_2 = z_{\text{med}} - z_{\max}$
- jika $A_1 > 0$ **dan** $A_2 < 0$ → median valid, lanjut ke Tahap B
- jika tidak → perbesar jendela; jika sudah $> S_{\max}$, keluarkan $z_{\text{med}}$

**Tahap B** — apakah piksel pusat sebuah impuls?
- $B_1 = z_{xy} - z_{\min}$, $B_2 = z_{xy} - z_{\max}$
- jika $B_1 > 0$ **dan** $B_2 < 0$ → piksel pusat bukan impuls, **keluarkan $z_{xy}$ tanpa diubah** (detail terjaga)
- jika tidak → keluarkan $z_{\text{med}}$

Kunci keunggulannya ada pada Tahap B: piksel yang sehat **tidak disentuh sama sekali**, sehingga ketajaman teks
bertahan meski jendela membesar hingga $7\times7$ di area yang rusak berat.

### Studi Kasus 9 — Berkas ijazah rusak berat: impuls 35%

Berkas unggahan pemohon rusak akibat kegagalan transfer; sepertiga pikselnya hancur. Median tetap sudah tidak
mampu; ini justru kasus yang menjadi alasan median adaptif diciptakan.

In [ ]:
def median_adaptif(img, s_max=7):
    """Implementasi tervektorisasi: median/min/maks untuk semua ukuran dihitung sekali,
    lalu logika Tahap A/B diterapkan sebagai operasi mask (jauh lebih cepat dari loop per piksel)."""
    assert s_max % 2 == 1
    h, w = img.shape
    pad = s_max // 2
    p = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REFLECT_101)
    iris = lambda a: a[pad:pad + h, pad:pad + w].astype(np.int16)

    out = img.copy()
    selesai = np.zeros((h, w), bool)
    z = img.astype(np.int16)

    for s in range(3, s_max + 1, 2):
        se = np.ones((s, s), np.uint8)
        zmed = iris(cv2.medianBlur(p, s))
        zmin = iris(cv2.erode(p, se))
        zmax = iris(cv2.dilate(p, se))

        A = (zmed > zmin) & (zmed < zmax)        # Tahap A: median bukan impuls
        B = (z > zmin) & (z < zmax)              # Tahap B: piksel pusat bukan impuls
        aktif = A & ~selesai
        out[aktif & B]  = img[aktif & B]          # piksel sehat -> dipertahankan apa adanya
        out[aktif & ~B] = zmed[aktif & ~B].astype(np.uint8)
        selesai |= aktif
        if selesai.all():
            break

    if (~selesai).any():                          # jendela maksimum tercapai -> pakai median terbesar
        zmed = iris(cv2.medianBlur(p, s_max))
        out[~selesai] = zmed[~selesai].astype(np.uint8)
    return out


RUSAK = derau_impulse(IJAZAH, p=0.35, seed=13)
import time
t = time.perf_counter(); ADAPT = median_adaptif(RUSAK, s_max=7); dt = time.perf_counter() - t

kand9 = [(RUSAK, "impuls 35%"),
         (cv2.medianBlur(RUSAK, 3), "median tetap 3x3"),
         (cv2.medianBlur(RUSAK, 5), "median tetap 5x5"),
         (cv2.medianBlur(RUSAK, 7), "median tetap 7x7"),
         (cv2.medianBlur(cv2.medianBlur(RUSAK, 3), 3), "median 3x3 dua kali"),
         (ADAPT, "median ADAPTIF (S_max=7)")]

tampil([(potong(k, "data"), n) for k, n in kand9], cols=2, tinggi=2.2,
       judul="Studi Kasus 9: impuls kerapatan tinggi")
print(f"waktu median adaptif: {dt*1000:.0f} ms untuk citra {RUSAK.shape}\n")
_ = bandingkan(IJAZAH, kand9)

In [ ]:
# Sejauh mana keunggulan bertahan saat kerapatan derau meningkat?
ps = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
m3, m7, ad = [], [], []
for p in ps:
    d = derau_impulse(IJAZAH, p=p, seed=21)
    m3.append(psnr(IJAZAH, cv2.medianBlur(d, 3), data_range=255))
    m7.append(psnr(IJAZAH, cv2.medianBlur(d, 7), data_range=255))
    ad.append(psnr(IJAZAH, median_adaptif(d, 7), data_range=255))

plt.figure(figsize=(6.4, 3.4))
plt.plot(ps, m3, "o-", label="median tetap 3x3")
plt.plot(ps, m7, "s-", label="median tetap 7x7")
plt.plot(ps, ad, "^-", label="median adaptif S_max=7")
plt.xlabel("kerapatan derau impuls p"); plt.ylabel("PSNR (dB)")
plt.title("Median adaptif vs median tetap"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("Titik silang median 3x3 dan 7x7 menunjukkan kapan kernel besar mulai diperlukan;")
print("median adaptif tidak memerlukan keputusan itu karena dibuat per piksel.")

> **Analisis.** Pada kerapatan rendah, median 3×3 sudah cukup dan median adaptif tidak memberi keuntungan berarti
> (jangan menambah kompleksitas tanpa alasan). Pada kerapatan tinggi, median tetap dihadapkan pada dilema:
> kernel kecil gagal membersihkan, kernel besar menghancurkan teks. Median adaptif memenangkan keduanya karena
> ukuran jendela ditentukan **per piksel** dan piksel sehat dilewatkan tanpa perubahan.
>
> **Catatan implementasi.** Versi loop per piksel pada citra 900×680 memerlukan puluhan detik di Python. Versi
> tervektorisasi di atas menghitung median/min/maks untuk semua ukuran jendela sekali saja lalu memakai *masking*,
> sehingga selesai dalam puluhan milidetik. Pola ini — ganti loop dengan operasi *array* + *mask* — berlaku umum
> untuk hampir semua filter adaptif.

---
## 10. Penajaman dengan Laplacian (Turunan Kedua)

### Teori

Penghalusan adalah integrasi; penajaman adalah **diferensiasi**. Laplacian adalah turunan kedua isotropik:

$$\nabla^{2} f = \frac{\partial^{2} f}{\partial x^{2}} + \frac{\partial^{2} f}{\partial y^{2}}$$

Dengan aproksimasi selisih hingga
$\partial^{2} f/\partial x^{2} = f(x+1,y) + f(x-1,y) - 2f(x,y)$, diperoleh kernel 4-tetangga; menyertakan
diagonal menghasilkan varian 8-tetangga:

$$w_{4} = \begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}, \qquad
w_{8} = \begin{bmatrix}1&1&1\\1&-8&1\\1&1&1\end{bmatrix}$$

Jumlah bobot = 0, sehingga area datar menjadi nol dan **hasil Laplacian bukan citra yang bisa ditampilkan
langsung** — nilainya bertanda dan berkisar jauh di luar [0, 255]. Untuk ditampilkan, geser dan skalakan.

Penajaman diperoleh dengan mengurangkan Laplacian dari citra asli (tanda mengikuti tanda bobot pusat):

$$g(x,y) = f(x,y) - c\,\nabla^{2} f(x,y), \qquad c = 1 \text{ bila pusat kernel negatif}$$

### Studi Kasus 10 — Ijazah kertas kusam: garis kop dan stempel pudar

Ijazah tahun 1990-an: tinta memudar, kontras rendah, garis tabel hampir menyatu dengan latar. Penajaman
Laplacian menegaskan kembali batas struktur.

In [ ]:
W4 = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float64)
W8 = np.array([[1, 1, 1], [1, -8, 1], [1, 1, 1]], np.float64)
tampil_kernel(W4, "Laplacian 4-tetangga", "{:+.0f}")
print()
tampil_kernel(W8, "Laplacian 8-tetangga", "{:+.0f}")

KUSAM = kertas_kusam(IJAZAH, 95, 205)

def tajamkan_laplacian(img, W=W4, c=1.0):
    f = img.astype(np.float64)
    lap = cv2.filter2D(f, cv2.CV_64F, W, borderType=cv2.BORDER_REFLECT_101)
    return (f - c * lap).clip(0, 255).astype(np.uint8), lap

def untuk_ditampilkan(lap):
    """Skala tanda-ganda ke 0-255 supaya bisa dilihat (nol -> abu-abu tengah)."""
    a = lap - lap.min()
    return (a / (a.max() + 1e-9) * 255).astype(np.uint8)

t4, lap4 = tajamkan_laplacian(KUSAM, W4, 1.0)
t8, lap8 = tajamkan_laplacian(KUSAM, W8, 1.0)
print(f"\nrentang nilai Laplacian: [{lap4.min():.0f}, {lap4.max():.0f}] -> harus diskalakan sebelum ditampilkan")

tampil([(potong(KUSAM, "kop"), "kusam (kontras rendah)"),
        (untuk_ditampilkan(potong(lap4, "kop")), "Laplacian w4 (diskalakan)"),
        (potong(t4, "kop"), "hasil tajam: f - lap (w4)"),
        (potong(t8, "kop"), "hasil tajam: f - lap (w8)")], cols=2, tinggi=2.0,
       judul="Studi Kasus 10: penajaman Laplacian pada kop ijazah pudar")

In [ ]:
# Pengaruh c: penajaman berlebih menghasilkan halo dan pemotongan (clipping)
varian = [(KUSAM, "c = 0 (asli)")] + [(tajamkan_laplacian(KUSAM, W4, c)[0], f"c = {c}") for c in [0.5, 1.0, 2.0, 4.0]]
tampil([(potong(v, "teks1"), n) for v, n in varian], cols=1, tinggi=1.45,
       judul="Pengaruh bobot penajaman c")

# Ukuran objektif ketajaman: energi gradien rata-rata + persentase piksel terpotong
print(f"{'Varian':<14}{'energi gradien':>16}{'piksel terpotong':>19}")
print("-" * 49)
for v, n in varian:
    gx = cv2.Sobel(v, cv2.CV_64F, 1, 0, ksize=3); gy = cv2.Sobel(v, cv2.CV_64F, 0, 1, ksize=3)
    energi = np.mean(np.hypot(gx, gy))
    potong_pct = np.mean((v == 0) | (v == 255)) * 100
    print(f"{n:<14}{energi:>16.2f}{potong_pct:>18.2f}%")
print("\nEnergi gradien naik terus, tetapi persentase piksel terpotong juga naik:")
print("informasi hilang permanen di area 0/255. Ketajaman tinggi tidak sama dengan kualitas tinggi.")

> **Analisis.** Laplacian bereaksi terhadap **derau** sama kuatnya dengan terhadap tepi, karena turunan kedua
> memperkuat frekuensi tinggi tanpa membedakan sumbernya. Karena itu aturan bakunya: **hilangkan derau dahulu,
> tajamkan kemudian**. Menajamkan citra berderau adalah kesalahan urutan pipeline yang paling sering ditemukan
> pada tugas mahasiswa. Varian 8-tetangga memberi respons diagonal sehingga terlihat lebih tajam, tetapi juga
> lebih rentan terhadap derau.

---
## 11. *Unsharp Masking* dan *High-Boost Filtering*

### Teori

Alih-alih memakai turunan kedua secara langsung, kita bisa memperoleh detail dengan **mengurangkan versi kabur**
dari citra asli. Prosedurnya tiga langkah:

1. Kaburkan: $\bar{f} = f * h_{\text{Gauss}}(\sigma)$
2. Bentuk *mask*: $g_{\text{mask}} = f - \bar{f}$  (berisi persis komponen frekuensi tinggi)
3. Tambahkan kembali dengan bobot: $g = f + k\, g_{\text{mask}}$

Dengan $k = 1$ disebut ***unsharp masking***; dengan $k > 1$ disebut ***high-boost filtering***.
Bila $k < 1$, detail justru ditekan.

Dua parameter dan artinya secara praktis:

- $\sigma$ menentukan **skala** detail yang diperkuat: $\sigma$ kecil menajamkan guratan huruf, $\sigma$ besar
  menaikkan kontras struktur besar (efek "kejelasan"/*clarity*).
- $k$ menentukan **kekuatan**. Terlalu besar → *halo* terang/gelap di sekitar tepi dan pemotongan nilai.

Metode ini lebih terkendali dari Laplacian karena $\sigma$ memisahkan skala detail dari kekuatan penajaman,
sehingga derau (skala sangat kecil) bisa dihindari.

### Studi Kasus 11 — Pindaian tidak fokus, dinilai dengan tiga metrik sekaligus

Kaca pemindai berdebu dan tutup dokumen tidak menekan rata: hasilnya kabur ($\sigma \approx 3$) dan OCR
kehilangan banyak karakter. Berapa nilai $k$ yang benar-benar memperbaiki keterbacaan — dan apakah PSNR, SSIM,
dan CER sepakat?

In [ ]:
def unsharp_masking(img, sigma=1.5, k=1.0):
    f = img.astype(np.float64)
    kabur = cv2.GaussianBlur(f, (0, 0), sigma)
    mask = f - kabur
    return (f + k * mask).clip(0, 255).astype(np.uint8), mask

KABUR = kabur_defokus(IJAZAH, sigma=3.0)
_, mask_demo = unsharp_masking(KABUR, sigma=2.5, k=1.0)

tampil([(potong(KABUR, "teks1"), "citra kabur (defokus)"),
        (untuk_ditampilkan(potong(mask_demo, "teks1")), "mask = f - blur(f) (diskalakan)"),
        (potong(unsharp_masking(KABUR, 2.5, 1.0)[0], "teks1"), "hasil: f + 1.0 * mask")],
       cols=1, tinggi=1.5, judul="Studi Kasus 11: tahapan unsharp masking")

In [ ]:
# Sapuan k, dinilai dengan PSNR dan CER sekaligus
print(f"{'k':>5}{'PSNR (dB)':>12}{'SSIM':>9}{'CER':>9}{'terpotong':>12}")
print("-" * 47)
baris = []
for k in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.5, 6.0, 8.0]:
    h, _ = unsharp_masking(KABUR, sigma=2.5, k=k)
    p, s = metrik(IJAZAH, h)
    cv_, _ = cer(potong(h, "data"), TEKS_ACUAN)
    tp = np.mean((h == 0) | (h == 255)) * 100
    baris.append((k, p, s, cv_, tp, h))
    print(f"{k:>5.1f}{p:>12.2f}{s:>9.4f}{cv_:>9.3f}{tp:>11.2f}%")
print(f"\n(CER acuan tanpa degradasi = {c0:.3f})")
opt_psnr = max(baris, key=lambda b: b[1])[0]
opt_ssim = max(baris, key=lambda b: b[2])[0]
opt_cer  = min(baris, key=lambda b: b[3])[0]
print(f"k optimum menurut PSNR = {opt_psnr} | menurut SSIM = {opt_ssim} | menurut CER = {opt_cer}")

tampil([(potong(b[5], "teks1"), f"k = {b[0]}  |  CER = {b[3]:.3f}") for b in baris],
       cols=1, tinggi=1.4, judul="Pengaruh k terhadap keterbacaan")

In [ ]:
# Pengaruh sigma pada k tetap: memilih SKALA detail yang diperkuat
tampil([(potong(unsharp_masking(KABUR, s, 1.5)[0], "teks1"), f"sigma = {s}, k = 1.5")
        for s in [0.8, 1.5, 2.5, 4.0, 7.0]], cols=1, tinggi=1.4,
       judul="sigma kecil menajamkan guratan; sigma besar menaikkan kontras struktur besar")

print("Catatan: unsharp masking TIDAK memulihkan informasi yang hilang karena kabur.")
print("Ia hanya menaikkan kontras lokal pada frekuensi yang masih tersisa. Pemulihan sejati")
print("memerlukan dekonvolusi (mis. Wiener), yang berada di luar lingkup domain spasial.")

> **Analisis — tiga metrik, tiga jawaban.** Perhatikan baris "k optimum" pada keluaran di atas: SSIM memilih
> $k$ kecil, PSNR memilih $k$ menengah, dan CER terus membaik sampai $k$ besar. Tidak ada yang keliru di sini;
> ketiganya mengukur hal yang berbeda:
>
> - SSIM menghukum perubahan struktur lokal, termasuk *halo* di sekitar guratan, sehingga cepat menurun.
> - PSNR menghukum galat kuadrat, dan mulai menurun begitu pemotongan (*clipping*) menjadi signifikan.
> - CER hanya peduli apakah karakter terpisah dan terbaca; kontras lokal yang sangat tinggi justru membantu
>   binarisasi internal Tesseract, sehingga CER masih membaik ketika citra sudah rusak menurut dua metrik lain.
>
> Kolom "terpotong" adalah penengahnya: pada $k$ besar, 10–15% piksel terdorong ke 0 atau 255, dan informasi
> di sana **hilang permanen** — merugikan setiap tahap hilir selain OCR (pencocokan foto, deteksi stempel).
> Rekomendasi kerja untuk dokumen: pilih $k$ **terkecil** yang sudah mendekati dataran CER, umumnya
> $k \in [1{,}5,\ 3]$ dengan $\sigma \in [1{,}5,\ 2{,}5]$, dan pantau persentase piksel terpotong sebagai
> pagar pengaman.

---
## 12. Filter Gradien: Roberts, Prewitt, Sobel, Scharr

### Teori

Turunan pertama citra adalah vektor gradien:

$$\nabla f = \begin{bmatrix} g_x \\ g_y \end{bmatrix} =
\begin{bmatrix} \partial f/\partial x \\ \partial f/\partial y \end{bmatrix}, \qquad
M(x,y) = \sqrt{g_x^{2}+g_y^{2}}, \qquad \alpha(x,y) = \arctan\frac{g_y}{g_x}$$

$M$ adalah **kekuatan tepi**, $\alpha$ adalah **arah tepi** (tegak lurus arah tepi itu sendiri). Dalam praktik
$M$ sering diaproksimasi $|g_x| + |g_y|$ agar hemat komputasi.

Empat pasang kernel yang umum:

$$\text{Roberts}: \begin{bmatrix}-1&0\\0&1\end{bmatrix},\ \begin{bmatrix}0&-1\\1&0\end{bmatrix} \qquad
\text{Prewitt}: \begin{bmatrix}-1&-1&-1\\0&0&0\\1&1&1\end{bmatrix},\ \begin{bmatrix}-1&0&1\\-1&0&1\\-1&0&1\end{bmatrix}$$

$$\text{Sobel}: \begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix},\ \begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix} \qquad
\text{Scharr}: \begin{bmatrix}-3&-10&-3\\0&0&0\\3&10&3\end{bmatrix},\ \begin{bmatrix}-3&0&3\\-10&0&10\\-3&0&3\end{bmatrix}$$

Perbedaan yang perlu dipahami, bukan dihafal:

| Kernel | Ukuran | Penghalusan | Karakter |
|---|---|---|---|
| Roberts | 2×2 | tidak ada | tercepat, paling sensitif derau, pusat tidak simetris |
| Prewitt | 3×3 | rerata seragam | seimbang, murah |
| Sobel | 3×3 | rerata berbobot (1,2,1) | standar de facto; tepi lebih terlokalisasi |
| Scharr | 3×3 | bobot (3,10,3) | rotasi paling akurat pada 3×3; dipakai bila arah gradien penting |

Bobot (1,2,1) pada Sobel **adalah** kernel Gaussian 1D diskret — jadi Sobel = penghalusan Gaussian arah
tegak lurus + diferensiasi, sedangkan Prewitt memakai penghalusan seragam (1,1,1). Perbedaan ini menentukan
**lokalisasi** tepi, bukan ketahanan derau: pengukuran di bawah menunjukkan Prewitt, Sobel, dan Scharr praktis
setara terhadap derau (selisih < 0,005), sementara Roberts tertinggal jelas karena tidak menghaluskan sama
sekali. Jadi alasan memilih Sobel sebagai baku adalah kombinasi lokalisasi, simetri, dan ketersediaan
implementasi — bukan keunggulan ketahanan derau atas Prewitt.

### Studi Kasus 12a — Koreksi kemiringan (*deskew*) sebelum OCR

Dokumen ijazah diletakkan miring di atas pemindai (8°). Tesseract punya koreksi kemiringan internal yang
menangani miring kecil (di bawah ±3° pengaruhnya nyaris nol), tetapi pada 8° akurasinya mulai jatuh.
Solusi: cari garis lurus dominan dengan gradien vertikal (`Sobel` $g_y$) → Hough → putar balik.

In [ ]:
# --- perbandingan keempat operator pada ROI kop ---
roi = potong(IJAZAH, "kop")

Rx = np.array([[-1, 0], [0, 1]], np.float64)
Ry = np.array([[0, -1], [1, 0]], np.float64)
Px = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], np.float64)
Py = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], np.float64)

def magnitudo(img, Kx, Ky):
    gx = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, Kx, borderType=cv2.BORDER_REFLECT_101)
    gy = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, Ky, borderType=cv2.BORDER_REFLECT_101)
    return np.hypot(gx, gy)

def norm255(a):
    return (a / (a.max() + 1e-9) * 255).astype(np.uint8)

m_rob = magnitudo(roi, Rx, Ry)
m_pre = magnitudo(roi, Px, Py)
m_sob = np.hypot(cv2.Sobel(roi, cv2.CV_64F, 1, 0, ksize=3), cv2.Sobel(roi, cv2.CV_64F, 0, 1, ksize=3))
m_sch = np.hypot(cv2.Scharr(roi, cv2.CV_64F, 1, 0), cv2.Scharr(roi, cv2.CV_64F, 0, 1))

tampil([(roi, "ROI kop (asli)"), (norm255(m_rob), "Roberts 2x2"), (norm255(m_pre), "Prewitt 3x3"),
        (norm255(m_sob), "Sobel 3x3"), (norm255(m_sch), "Scharr 3x3")], cols=1, tinggi=1.8,
       judul="Perbandingan operator gradien")

# ketahanan terhadap derau: korelasi peta tepi bersih vs peta tepi berderau
BERDERAU = derau_gaussian(IJAZAH, sigma=15, seed=31)
roi_n = potong(BERDERAU, "kop")
print(f"{'Operator':<12}{'korelasi peta tepi bersih vs berderau':>40}")
print("-" * 52)
for nama, f in [("Roberts", lambda a: magnitudo(a, Rx, Ry)),
                ("Prewitt", lambda a: magnitudo(a, Px, Py)),
                ("Sobel",   lambda a: np.hypot(cv2.Sobel(a, cv2.CV_64F, 1, 0, ksize=3), cv2.Sobel(a, cv2.CV_64F, 0, 1, ksize=3))),
                ("Scharr",  lambda a: np.hypot(cv2.Scharr(a, cv2.CV_64F, 1, 0), cv2.Scharr(a, cv2.CV_64F, 0, 1)))]:
    a, b = f(roi).ravel(), f(roi_n).ravel()
    print(f"{nama:<12}{np.corrcoef(a, b)[0, 1]:>40.4f}")
print("\nMakin tinggi korelasi, makin stabil operator itu terhadap derau.")

In [ ]:
# --- Studi Kasus 12a: deskew berbasis Sobel + Hough ---
def putar(img, sudut, latar=246):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), sudut, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=latar)

def estimasi_kemiringan(img, ambang_gradien=200):
    """Deteksi garis horizontal dominan lewat gradien vertikal, lalu ambil median sudutnya."""
    gy = cv2.Sobel(cv2.GaussianBlur(img, (3, 3), 0), cv2.CV_64F, 0, 1, ksize=3)
    tepi = (np.abs(gy) > ambang_gradien).astype(np.uint8) * 255
    garis = cv2.HoughLinesP(tepi, 1, np.pi / 360, threshold=120, minLineLength=250, maxLineGap=12)
    if garis is None:
        return 0.0, tepi, 0
    sudut = [np.degrees(np.arctan2(y2 - y1, x2 - x1)) for x1, y1, x2, y2 in garis[:, 0]]
    sudut = [s for s in sudut if abs(s) < 15]          # buang garis vertikal/diagonal
    return (float(np.median(sudut)) if sudut else 0.0), tepi, len(sudut)

MIRING = putar(IJAZAH, -8.0)
est, tepi, n = estimasi_kemiringan(MIRING)
LURUS = putar(MIRING, est)

print(f"kemiringan sebenarnya : -8.00 derajat")
print(f"kemiringan terestimasi: {est:+.2f} derajat  (dari {n} segmen garis)")
print(f"galat                 : {abs(abs(est) - 8.0):.2f} derajat")

tampil([(potong(MIRING, "kop"), "miring (input)"),
        (potong(tepi, "kop"), "peta |Sobel gy| > ambang"),
        (potong(LURUS, "kop"), "setelah deskew")], cols=1, tinggi=1.8,
       judul="Studi Kasus 12a: deskew dengan gradien vertikal + Hough")

if OCR_ADA:
    print(f"\nCER miring       : {cer(potong(MIRING, 'data'), TEKS_ACUAN)[0]:.3f}")
    print(f"CER setelah deskew: {cer(potong(LURUS, 'data'), TEKS_ACUAN)[0]:.3f}")
    print(f"CER acuan         : {c0:.3f}")

### Studi Kasus 12b — Lokalisasi tanda tangan pada blok pengesahan

Verifikasi ijazah membutuhkan pengecekan keberadaan tanda tangan. Persoalannya: pada ROI yang sama terdapat
teks cetak, garis, dan stempel. Ciri pembeda tanda tangan bukan intensitas atau ukuran, melainkan **kerapatan
tinta di dalam kotak pembatasnya**: tanda tangan adalah guratan panjang melengkung yang menempati kotak besar
dengan isi renggang, sedangkan teks cetak mengisi kotaknya jauh lebih padat.

Pipeline: Sobel → ambang → morfologi *closing* (menyatukan guratan yang terputus) → komponen terhubung →
seleksi berdasarkan (luas, rasio isi).

In [ ]:
def deteksi_tanda_tangan(img_roi, ambang=70, min_area=700, kernel_tutup=(13, 7)):
    g = cv2.GaussianBlur(img_roi, (3, 3), 0)
    mag = np.hypot(cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3))
    mag = norm255(mag)
    bw = (mag > ambang).astype(np.uint8) * 255
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, kernel_tutup))

    n, lab, stats, _ = cv2.connectedComponentsWithStats(bw, 8)
    komp = []
    for i in range(1, n):
        x, y, w, h, a = (int(v) for v in stats[i])
        if a < min_area:
            continue
        komp.append({"kotak": (x, y, w, h), "luas": a, "rasio_isi": a / (w * h),
                     "rasio_aspek": w / max(h, 1)})
    komp.sort(key=lambda k: -k["luas"])
    return bw, mag, komp, n - 1        # n-1 = total komponen sebelum penyaringan luas


def klasifikasi_ttd(k, min_luas=5000, maks_isi=0.35):
    """Tanda tangan = kotak besar dengan isi renggang. Teks cetak mengisi kotaknya jauh lebih padat."""
    return k["luas"] > min_luas and k["rasio_isi"] < maks_isi


roi_ttd = potong(IJAZAH, "ttd")
bw, mag, komp, n_total = deteksi_tanda_tangan(roi_ttd)

print(f"total komponen terhubung sebelum penyaringan luas: {n_total}\n")
print(f"{'#':<3}{'kotak (x,y,w,h)':<26}{'luas':>8}{'rasio isi':>11}{'aspek':>8}   dugaan")
print("-" * 74)
for i, k in enumerate(komp[:6]):
    dugaan = "TANDA TANGAN" if klasifikasi_ttd(k) else "teks/garis cetak"
    print(f"{i:<3}{str(k['kotak']):<26}{k['luas']:>8}{k['rasio_isi']:>11.2f}{k['rasio_aspek']:>8.2f}   {dugaan}")

anotasi = cv2.cvtColor(roi_ttd, cv2.COLOR_GRAY2BGR)
for k in komp[:6]:
    x, y, w, h = k["kotak"]
    cv2.rectangle(anotasi, (x, y), (x + w, y + h), (0, 200, 0) if klasifikasi_ttd(k) else (255, 120, 0), 2)

fig, ax = plt.subplots(1, 3, figsize=(13, 2.8))
for a, (im, t) in zip(ax, [(mag, "magnitudo Sobel"), (bw, "ambang + closing"),
                           (cv2.cvtColor(anotasi, cv2.COLOR_BGR2RGB), "hijau = tanda tangan terdeteksi")]):
    a.imshow(im, cmap=None if im.ndim == 3 else "gray"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Ketahanan pipeline: apakah deteksi tetap bekerja pada citra berderau, dan apa efek pemfilteran?
varian = [("acuan", roi_ttd),
          ("impuls 8%", potong(derau_impulse(IJAZAH, 0.08, seed=1), "ttd")),
          ("impuls 8% + median 3x3", potong(cv2.medianBlur(derau_impulse(IJAZAH, 0.08, seed=1), 3), "ttd"))]
print(f"{'Varian':<26}{'total CC':>10}{'CC besar':>10}{'luas terbesar':>15}{'rasio isi':>11}")
print("-" * 72)
for nama, im in varian:
    _, _, k, n_tot = deteksi_tanda_tangan(im)
    if k:
        print(f"{nama:<26}{n_tot:>10}{len(k):>10}{k[0]['luas']:>15}{k[0]['rasio_isi']:>11.2f}")
    else:
        print(f"{nama:<26}{n_tot:>10}{0:>10}{'-':>15}{'-':>11}")
print("\nDerau impuls meledakkan jumlah komponen (puluhan -> ratusan) dan, lebih buruk lagi,")
print("menyambungkan tanda tangan dengan stempel serta teks menjadi satu blob raksasa:")
print("kotak pembatas terbesar membengkak jauh melampaui ukuran tanda tangan sebenarnya,")
print("sehingga deteksi 'berhasil' dengan kotak yang salah. Median 3x3 memulihkan strukturnya.")
print("Inilah alasan konkret mengapa urutan denoise -> gradien wajib dipatuhi.")

> **Analisis.** Filter gradien **bukan** filter peningkatan kualitas dalam arti kosmetik — keluarannya bukan
> citra yang lebih enak dilihat, melainkan **peta fitur** untuk tahap berikutnya (Hough, komponen terhubung,
> kontur). Karena turunan memperkuat derau, filter gradien hampir selalu diawali penghalusan; `cv2.Sobel`
> sendiri sudah mengandung penghalusan (1,2,1), dan itulah sebabnya ia dipilih sebagai baku ketimbang Roberts.

---
## 13. Filter Bilateral

### Teori

Semua filter linear di Bab 4–5 punya satu cacat mendasar: bobot hanya bergantung pada **jarak spasial**, sehingga
piksel di seberang tepi ikut dirata-ratakan dan tepi menjadi kabur. Filter bilateral menambahkan faktor kedua
yang bergantung pada **selisih intensitas**:

$$g(x,y) = \frac{1}{W}\sum_{(s,t)\in S_{xy}}
\underbrace{e^{-\frac{(x-s)^2+(y-t)^2}{2\sigma_{d}^{2}}}}_{\text{kedekatan spasial}}
\cdot
\underbrace{e^{-\frac{\left(f(x,y)-f(s,t)\right)^{2}}{2\sigma_{r}^{2}}}}_{\text{kemiripan intensitas}}
\cdot f(s,t)$$

dengan $W$ = jumlah seluruh bobot (normalisasi). Karena bobot bergantung pada nilai piksel, filter ini
**non-linear** dan tidak bisa dinyatakan sebagai satu kernel tetap — kernel efektifnya berbeda di setiap piksel.

Interpretasi parameter (`cv2.bilateralFilter(src, d, sigmaColor, sigmaSpace)`):

- `sigmaSpace` ($\sigma_d$): seberapa jauh piksel boleh berpengaruh (satuan piksel).
- `sigmaColor` ($\sigma_r$): seberapa besar selisih intensitas yang masih dianggap "sama permukaan".
  Ini parameter kuncinya. $\sigma_r \gg$ kontras tepi → berperilaku seperti Gaussian biasa (tepi ikut kabur);
  $\sigma_r \ll$ amplitudo derau → nyaris tidak menghaluskan apa pun.
- `d`: diameter jendela; `d=0` berarti dihitung dari `sigmaSpace`.

### Studi Kasus 13 — Derau pada foto pas 3×4 di ijazah

Foto pas pada ijazah perlu dihaluskan (derau ISO tinggi) tanpa mengaburkan **kontur wajah dan garis kerah**,
karena kontur itu dipakai tahap pencocokan wajah. Inilah kasus kanonik filter bilateral.

In [ ]:
FOTO_DERAU = derau_gaussian(IJAZAH, sigma=22, seed=41)
roi_ref  = potong(IJAZAH, "foto")
roi_uji  = potong(FOTO_DERAU, "foto")

kand13 = [(roi_uji, "berderau (sigma=22)"),
          (cv2.GaussianBlur(roi_uji, (0, 0), 1.6), "Gaussian sigma=1.6"),
          (cv2.medianBlur(roi_uji, 5), "median 5x5"),
          (cv2.bilateralFilter(roi_uji, 9, 40, 9), "bilateral d=9, sC=40"),
          (cv2.bilateralFilter(roi_uji, 9, 75, 9), "bilateral d=9, sC=75"),
          (cv2.bilateralFilter(roi_uji, 9, 150, 9), "bilateral d=9, sC=150")]

tampil(kand13, cols=3, tinggi=2.6, judul="Studi Kasus 13: penghalusan foto pas tanpa merusak tepi")
print()
_ = bandingkan(roi_ref, kand13)

In [ ]:
# Bukti kuantitatif "tepi terjaga": profil intensitas menurun melintasi batas rambut/wajah/kerah
kol = roi_ref.shape[1] // 2      # kolom tengah: memotong rambut, wajah, dan garis kerah
plt.figure(figsize=(7.2, 3.4))
plt.plot(roi_ref[:, kol], "k-", lw=2.0, label="acuan (tepi ideal)")
plt.plot(roi_uji[:, kol], color="0.75", lw=0.9, label="berderau")
plt.plot(cv2.GaussianBlur(roi_uji, (0, 0), 1.6)[:, kol], "--", label="Gaussian s=1.6")
plt.plot(cv2.bilateralFilter(roi_uji, 9, 75, 9)[:, kol], "-", label="bilateral sC=75")
plt.xlabel("posisi baris"); plt.ylabel("intensitas"); plt.title(f"Profil vertikal pada kolom {kol}")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

# Ketajaman tepi diukur dari kemiringan maksimum lintas-tepi
def ketajaman_tepi(img, kol):
    return np.abs(np.diff(img[:, kol].astype(np.float64))).max()

print(f"{'Citra':<24}{'kemiringan tepi maks':>22}")
print("-" * 46)
for im, nama in [(roi_ref, "acuan"), (roi_uji, "berderau"),
                 (cv2.GaussianBlur(roi_uji, (0, 0), 1.6), "Gaussian s=1.6"),
                 (cv2.bilateralFilter(roi_uji, 9, 75, 9), "bilateral sC=75")]:
    print(f"{nama:<24}{ketajaman_tepi(im, kol):>22.1f}")
print("\nGaussian menurunkan kemiringan tepi (tepi melandai = kabur);")
print("bilateral mempertahankannya mendekati acuan sambil tetap menghaluskan area datar.")

> **Analisis.** Bilateral menang pada citra bergaya foto, tetapi **bukan** pilihan utama untuk teks dokumen:
> guratan huruf tipis memiliki lebar hanya 1–2 piksel sehingga sering dianggap "tepi" dan derau di sekitarnya
> justru dipertahankan. Selain itu biayanya jauh lebih tinggi daripada Gaussian. Gunakan bilateral untuk
> **wilayah foto** di dalam dokumen, dan median/Gaussian untuk wilayah teks — pipeline nyata memang memfilter
> ROI yang berbeda dengan filter yang berbeda.

---
## 14. *Non-Local Means* (NLM)

### Teori

Bilateral membandingkan **satu piksel** dengan piksel pusat. NLM membandingkan **seluruh tampalan (*patch*)**
di sekitarnya, dan boleh mengambil tampalan mirip dari **mana saja** dalam jendela pencarian — bukan hanya
tetangga terdekat:

$$g(x) = \frac{1}{W(x)}\sum_{y \in \Omega(x)} e^{-\frac{\lVert P(x)-P(y)\rVert^{2}}{h^{2}}} f(y)$$

dengan $P(x)$ = tampalan berpusat di $x$ dan $h$ = parameter penyaring (sebanding dengan $\sigma$ derau).

Asumsinya: citra alami **mengandung pengulangan**. Pada dokumen, asumsi ini sangat kuat — huruf "a" yang sama
muncul puluhan kali, garis tabel berulang panjang. Karena itu NLM biasanya mengalahkan bilateral pada citra
dokumen berderau, dengan biaya komputasi yang jauh lebih besar.

### Studi Kasus 14 — Arsip ijazah hasil pindai mikrofilm, derau kuat ($\sigma = 30$)

In [ ]:
from skimage.restoration import denoise_nl_means

SIGMA = 30
NOISY = derau_gaussian(IJAZAH, sigma=SIGMA, seed=51)
roi_ref, roi_uji = potong(IJAZAH, "data"), potong(NOISY, "data")

import time
t = time.perf_counter()
nlm = denoise_nl_means(roi_uji / 255.0, h=1.15 * SIGMA / 255.0, sigma=SIGMA / 255.0,
                       fast_mode=True, patch_size=5, patch_distance=6)
t_nlm = time.perf_counter() - t
nlm = (nlm * 255).clip(0, 255).astype(np.uint8)

t = time.perf_counter(); bil = cv2.bilateralFilter(roi_uji, 9, 75, 9); t_bil = time.perf_counter() - t
t = time.perf_counter(); gau = cv2.GaussianBlur(roi_uji, (0, 0), 1.2);  t_gau = time.perf_counter() - t

kand14 = [(roi_uji, "berderau sigma=30"), (gau, "Gaussian s=1.2"),
          (cv2.medianBlur(roi_uji, 3), "median 3x3"), (bil, "bilateral"), (nlm, "non-local means")]
tampil(kand14, cols=1, tinggi=2.0, judul="Studi Kasus 14: derau kuat pada arsip pindaian")
print()
_ = bandingkan(roi_ref, kand14)
print(f"\nWaktu komputasi ROI {roi_uji.shape}:  Gaussian {t_gau*1000:.1f} ms | "
      f"bilateral {t_bil*1000:.1f} ms | NLM {t_nlm*1000:.1f} ms")

> **Analisis.** NLM biasanya memberi PSNR/SSIM tertinggi di antara semua filter pada notebook ini, tetapi
> harganya adalah waktu komputasi satu hingga dua orde lebih besar. Untuk sistem admisi yang memproses ribuan
> unggahan per hari, NLM hanya layak dipakai **pada ROI kecil yang kritis** (blok nomor ijazah, tanda tangan),
> bukan pada seluruh halaman. Parameter `h` harus sebanding dengan $\sigma$ derau; menaksir `h` terlalu besar
> menghasilkan citra yang tampak "diplastikkan" (*over-smoothed*).

---
## 15. Filter Adaptif Lokal untuk Iluminasi Tidak Rata

### Teori

Semua filter sebelumnya memakai parameter **global**. Ketika degradasinya bervariasi menurut posisi — lampu
pindai lebih terang di satu sisi — filter berparameter global pasti gagal: apa pun ambang yang dipilih untuk
binarisasi, satu sisi halaman akan salah.

Tiga pendekatan spasial-lokal yang saling melengkapi:

**(1) Pengurangan latar belakang.** Latar (iluminasi) adalah komponen frekuensi sangat rendah. Estimasi dengan
filter rerata/median **berkernel besar**, lalu bagi (karena degradasinya multiplikatif):

$$\hat{b} = f * h_{\text{rerata besar}}, \qquad g = \frac{f}{\hat{b}} \cdot \bar{b}$$

**(2) Peningkatan statistik lokal.** Ganti parameter global dengan statistik ketetanggaan — rerata lokal
$m_{S_{xy}}$ dan simpangan baku lokal $\sigma_{S_{xy}}$ — yang dihitung efisien dengan *box filter*:

$$\sigma_{S_{xy}}^{2} = \overline{f^{2}} - \bar{f}^{2}$$

**(3) Pengambangan adaptif.** Ambang ditentukan per piksel dari rerata (atau rerata berbobot Gaussian) lokal:
$T(x,y) = m_{S_{xy}} - C$. Ini persis `cv2.adaptiveThreshold`, dan secara mekanis adalah **filter spasial
diikuti perbandingan**.

### Studi Kasus 15 — Ijazah dipindai dengan lampu lemah di satu sisi, dinilai dengan CER OCR

In [ ]:
TIMPANG = iluminasi_tidak_rata(IJAZAH, kuat=0.70)

K_LATAR = int(101 * SKALA) | 1        # kernel estimasi latar, harus jauh lebih besar dari tinggi huruf
K_BLOK  = int(31 * SKALA) | 1         # blockSize ambang adaptif, harus lebih besar dari tinggi huruf

def koreksi_latar(img, k=K_LATAR, metode="rerata"):
    """Estimasi iluminasi dengan kernel besar lalu bagi (degradasi multiplikatif)."""
    f = img.astype(np.float64)
    if metode == "rerata":
        latar = cv2.blur(f, (k, k))
    else:                                     # median lebih tahan terhadap teks pekat
        latar = cv2.medianBlur(img, k if k % 2 else k + 1).astype(np.float64)
    latar = np.maximum(latar, 1.0)
    return (f / latar * latar.mean()).clip(0, 255).astype(np.uint8), latar

KOREKSI, latar = koreksi_latar(TIMPANG, K_LATAR, "rerata")
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(TIMPANG)

tampil([(TIMPANG, "iluminasi tidak rata"),
        (latar.astype(np.uint8), f"estimasi latar (rerata {K_LATAR}x{K_LATAR})"),
        (KOREKSI, "hasil pembagian latar"),
        (CLAHE, "CLAHE (histogram lokal)")], cols=2, tinggi=2.6,
       judul="Studi Kasus 15: koreksi iluminasi berbasis filter berkernel besar")

In [ ]:
# Statistik lokal dihitung efisien dengan box filter (bukan loop)
def statistik_lokal(img, k=25):
    f = img.astype(np.float64)
    m = cv2.boxFilter(f, -1, (k, k))
    v = cv2.boxFilter(f * f, -1, (k, k)) - m * m
    return m, np.sqrt(np.maximum(v, 0))

m_lok, s_lok = statistik_lokal(TIMPANG, int(25 * SKALA) | 1)
print("rerata lokal: rentang %.1f - %.1f  (jelas bervariasi -> parameter global tidak memadai)"
      % (m_lok.min(), m_lok.max()))
tampil([(norm255(m_lok), "rerata lokal (peta iluminasi)"),
        (norm255(s_lok), "simpangan baku lokal (peta 'ada detail')")], cols=2, tinggi=2.6)

In [ ]:
# Binarisasi: ambang global vs adaptif, dinilai dengan CER
_, otsu = cv2.threshold(TIMPANG, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
ad_mean = cv2.adaptiveThreshold(TIMPANG, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, K_BLOK, 10)
ad_gaus = cv2.adaptiveThreshold(TIMPANG, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, K_BLOK, 10)
_, otsu_setelah_koreksi = cv2.threshold(KOREKSI, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

kand15 = [(TIMPANG, "iluminasi timpang (input)"),
          (otsu, "Otsu global"),
          (ad_mean, f"adaptif MEAN_C {K_BLOK}, C=10"),
          (ad_gaus, f"adaptif GAUSSIAN_C {K_BLOK}, C=10"),
          (KOREKSI, "koreksi latar (grayscale)"),
          (otsu_setelah_koreksi, "koreksi latar + Otsu"),
          (CLAHE, "CLAHE")]

tampil([(potong(k, "data"), n) for k, n in kand15], cols=1, tinggi=1.9,
       judul="Ambang global gagal di sisi gelap; ambang adaptif tidak")

print(f"{'Metode':<34}{'CER':>8}")
print("-" * 42)
print(f"{'ACUAN (tanpa degradasi)':<34}{c0:>8.3f}")
for img, nama in kand15:
    print(f"{nama:<34}{cer(potong(img, 'data'), TEKS_ACUAN)[0]:>8.3f}")

# Mengapa Otsu global gagal: proporsi piksel yang masih dinilai "kertas" per band horizontal
print("\nProporsi piksel yang diklasifikasikan sebagai KERTAS (bukan tinta), per band horizontal:")
h = TIMPANG.shape[0]
print(f"  {'band':<10}{'Otsu global':>14}{'ambang adaptif':>17}")
for i in range(3):
    y0, y1 = i * h // 3, (i + 1) * h // 3
    print(f"  {f'band {i+1}':<10}{otsu[y0:y1].mean()/255:>14.3f}{ad_gaus[y0:y1].mean()/255:>17.3f}")
print("  Pada band tergelap, Otsu global menyatakan hampir seluruh kertas sebagai tinta.")

> **Analisis.** Otsu global memilih satu ambang untuk seluruh halaman; pada band tergelap, kertas dinilai
> sebagai tinta sehingga hampir seluruh blok menjadi hitam dan OCR runtuh. Dua solusi tersedia, dan keduanya
> **filter spasial**: (a) memperbaiki citranya lebih dahulu — pembagian latar dengan kernel besar, setelah itu
> Otsu global bekerja normal karena iluminasinya sudah seragam; atau (b) membuat ambangnya lokal
> (`adaptiveThreshold`). Pada percobaan ini keduanya mencapai CER yang setara dan jauh di bawah Otsu global.
>
> Perhatikan juga bahwa CER citra grayscale yang timpang **tidak** seburuk versi biner Otsu-nya. Sebabnya:
> Tesseract melakukan binarisasi lokalnya sendiri. Pelajaran praktisnya — jangan serahkan citra yang sudah
> dibinarisasi secara global ke mesin OCR; berikan grayscale, atau binarisasi dengan metode lokal.
>
> Catatan: ukuran blok pada `adaptiveThreshold` harus **lebih besar dari objek yang ingin dipertahankan**
> (di sini tinggi huruf). Blok terlalu kecil membuat bagian dalam guratan tebal ikut dianggap latar dan huruf
> menjadi berlubang.

---
## 16. Studi Kasus Terpadu: Pipeline Lengkap

Sekarang seluruh degradasi hadir bersamaan pada satu berkas — persis kondisi berkas unggahan nyata:

1. iluminasi tidak rata (lampu pindai lemah di satu sisi),
2. derau impuls 4% (sensor rusak),
3. derau Gaussian $\sigma = 6$ (elektronik),
4. defokus ringan $\sigma = 0{,}8$,
5. miring 5°.

**Urutan pipeline bukan sembarangan.** Aturan kerja yang dipakai di sini:

> **derau impuls → geometri → derau aditif → koreksi iluminasi → (penajaman) → binarisasi**

Alasan tiap panah: median dijalankan **sebelum** rotasi, karena interpolasi rotasi mengubah bintik impuls
menjadi corengan abu-abu yang tidak lagi berupa nilai ekstrem — dan di sana median kehilangan keunggulannya;
penajaman diletakkan setelah semua penghalusan karena ia memperkuat derau; binarisasi selalu terakhir karena
membuang informasi tingkat keabuan yang masih dibutuhkan tahap sebelumnya.

Aturan seperti ini beredar luas, dan justru karena itu perlu **diuji, bukan diwarisi**. Alih-alih menyatakan
bahwa setiap tahap "membantu", kita mengukur kontribusi masing-masing dengan **ablasi**: matikan satu tahap,
lihat perubahan CER-nya. Beberapa klaim akan terbukti, beberapa tidak.

In [ ]:
def buat_berkas_rusak(src, seed=61):
    x = iluminasi_tidak_rata(src, kuat=0.40)
    x = derau_impulse(x, p=0.04, seed=seed)
    x = derau_gaussian(x, sigma=6, seed=seed + 1)
    x = kabur_defokus(x, sigma=0.8)
    return putar(x, -5.0, latar=215)


BERKAS = buat_berkas_rusak(IJAZAH)


def pipeline(img):
    tahap = [("0. input", img)]

    x = cv2.medianBlur(img, 3)                                    # 1. derau impuls, SEBELUM interpolasi
    tahap.append(("1. median 3x3 (impuls)", x))

    sudut, _, _ = estimasi_kemiringan(x, ambang_gradien=150)      # 2. geometri
    x = putar(x, sudut, latar=215)
    tahap.append((f"2. deskew ({sudut:+.2f} derajat)", x))

    x = cv2.GaussianBlur(x, (0, 0), 0.6)                          # 3. derau aditif sisa
    tahap.append(("3. Gaussian s=0.6 (aditif)", x))

    x, _ = koreksi_latar(x, K_LATAR, "rerata")                    # 4. iluminasi
    tahap.append(("4. koreksi latar", x))

    _, b = cv2.threshold(x, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # 5. binarisasi
    tahap.append(("5. Otsu global", b))
    return x, b, tahap


HASIL, BINER, tahap = pipeline(BERKAS)
tampil([(potong(im, "data"), n) for n, im in tahap], cols=1, tinggi=1.9,
       judul="Pipeline peningkatan kualitas, tahap demi tahap (ROI data)")

In [ ]:
print(f"{'Tahap':<30}{'PSNR':>8}{'SSIM':>8}{'CER':>8}")
print("-" * 54)
for n, im in tahap:
    p, sv = psnr(IJAZAH, im, data_range=255), ssim(IJAZAH, im, data_range=255)
    c_, _ = cer(potong(im, "data"), TEKS_ACUAN)
    print(f"{n:<30}{p:>8.2f}{sv:>8.3f}{c_:>8.3f}")
print(f"{'ACUAN (tanpa degradasi)':<30}{'inf':>8}{1.0:>8.3f}{c0:>8.3f}")

print("\nHasil OCR SEBELUM pipeline:")
print("  ", cer(potong(BERKAS, "data"), TEKS_ACUAN)[1][:170])
print("Hasil OCR SETELAH pipeline (grayscale):")
print("  ", cer(potong(HASIL, "data"), TEKS_ACUAN)[1][:170])
print("Hasil OCR SETELAH binarisasi:")
print("  ", cer(potong(BINER, "data"), TEKS_ACUAN)[1][:170])

> **Perhatikan kolom PSNR.** Nilainya bertahan rendah dari awal sampai akhir, padahal CER turun lebih dari
> sepuluh kali lipat. Ini bukan kegagalan pipeline melainkan **kegagalan metrik**: degradasi iluminasi bersifat
> multiplikatif, dan koreksi latar memetakan ulang seluruh tingkat keabuan. PSNR menghukum pemetaan itu sebagai
> galat besar di setiap piksel, sementara untuk OCR yang penting hanyalah kontras relatif antara tinta dan
> kertas. Untuk pipeline yang memuat transformasi intensitas, PSNR bukan metrik yang sah.

In [ ]:
# ABLASI: matikan satu tahap, ukur perubahan CER
def rantai(median=True, deskew=True, gauss=True, latar=True, unsharp=False, ambang="otsu"):
    y = BERKAS
    if median:
        y = cv2.medianBlur(y, 3)
    if deskew:
        y = putar(y, estimasi_kemiringan(y, ambang_gradien=150)[0], latar=215)
    if gauss:
        y = cv2.GaussianBlur(y, (0, 0), 0.6)
    if latar:
        y, _ = koreksi_latar(y, K_LATAR, "rerata")
    if unsharp:
        y, _ = unsharp_masking(y, sigma=1.0, k=0.5)
    if ambang == "otsu":
        _, y = cv2.threshold(y, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif ambang == "adaptif":
        y = cv2.adaptiveThreshold(y, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, K_BLOK, 10)
    return y


def urutan_salah():
    """Penajaman dijalankan lebih dahulu, ketika derau masih penuh."""
    y, _ = unsharp_masking(BERKAS, sigma=1.0, k=0.5)
    y = cv2.medianBlur(y, 3)
    y = putar(y, estimasi_kemiringan(y, ambang_gradien=150)[0], latar=215)
    y, _ = koreksi_latar(y, K_LATAR, "rerata")
    _, y = cv2.threshold(y, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return y


LENGKAP = rantai()
c_lengkap = cer(potong(LENGKAP, "data"), TEKS_ACUAN)[0]
ablasi = [("input tanpa perlakuan", BERKAS),
          ("pipeline LENGKAP", LENGKAP),
          ("tanpa median", rantai(median=False)),
          ("tanpa deskew", rantai(deskew=False)),
          ("tanpa Gaussian", rantai(gauss=False)),
          ("tanpa koreksi latar", rantai(latar=False)),
          ("+ unsharp (s=1.0, k=0.5)", rantai(unsharp=True)),
          ("ambang adaptif, bukan Otsu", rantai(ambang="adaptif")),
          ("tanpa binarisasi (grayscale)", rantai(ambang=None)),
          ("urutan lain: tajamkan dahulu", urutan_salah())]

# Klaim yang diuji: penajaman memperkuat derau impuls. Ukur porsi piksel ekstrem sebelum/sesudah.
ekstrem = lambda a: np.mean((a == 0) | (a == 255)) * 100
print(f"porsi piksel ekstrem (0 atau 255) pada berkas rusak: {ekstrem(BERKAS):.2f}%")
for sg, kk in [(1.0, 0.5), (1.5, 1.5), (2.0, 2.5)]:
    t, _ = unsharp_masking(BERKAS, sigma=sg, k=kk)
    print(f"  setelah unsharp (s={sg}, k={kk}): {ekstrem(t):.2f}%")
print()

print(f"{'Varian':<32}{'CER':>8}{'selisih vs lengkap':>21}")
print("-" * 61)
for nama, im in ablasi:
    v = cer(potong(im, "data"), TEKS_ACUAN)[0]
    tanda = "(acuan)" if nama == "pipeline LENGKAP" else f"{v - c_lengkap:+.3f}"
    print(f"{nama:<32}{v:>8.3f}{tanda:>21}")

tampil([(potong(BERKAS, "teks1"), "input rusak"),
        (potong(rantai(latar=False), "teks1"), "tanpa koreksi latar"),
        (potong(urutan_salah(), "teks1"), "urutan salah: tajamkan dahulu"),
        (potong(LENGKAP, "teks1"), "pipeline lengkap")], cols=1, tinggi=1.45)

> **Analisis ablasi.** Angka-angkanya memberi tiga pelajaran yang tidak bisa diperoleh dari teori saja:
>
> 1. **Koreksi iluminasi adalah penyumbang terbesar** — mematikannya melonjakkan CER berkali-kali lipat, jauh
>    melebihi dampak mematikan median. Pada dokumen pindai, masalah pencahayaan sering lebih merusak daripada
>    derau, meskipun derau yang lebih mencolok secara visual.
> 2. **Tidak semua tahap berkontribusi positif.** Penghalusan Gaussian tambahan hampir tidak berguna di sini
>    (median sudah menangani deraunya) dan penajaman hanya memberi perubahan marginal. Tahap yang tidak
>    terbukti berkontribusi sebaiknya **dibuang**, bukan dipertahankan karena "biasanya ada di pipeline".
> 3. **Setelah iluminasi dikoreksi, Otsu global mengalahkan ambang adaptif.** Ini konsisten dengan Bab 15:
>    ambang adaptif adalah obat untuk iluminasi tidak rata; begitu penyakitnya sudah disembuhkan, jendela
>    lokalnya justru menambah kepekaan terhadap derau sisa. Menumpuk dua solusi untuk satu masalah tidak
>    membuat hasilnya dua kali lebih baik.
>
> **Dan satu klaim yang tidak terbukti.** Aturan "denoise sebelum menajamkan" punya dasar fisik yang nyata,
> tetapi pengukuran porsi piksel ekstrem menunjukkan dasar itu **bergantung pada kekuatan penajaman**: pada
> $k = 0{,}5$ tidak ada piksel yang terdorong ke batas 0/255, dan penguatan impuls baru terlihat pada $k$ yang
> lebih besar. Konsisten dengan itu, selisih CER antara kedua urutan pada berkas ini berada dalam rentang derau
> pengukuran (±0,03) — menukar urutannya tidak merugikan. Dua sebab tambahan: (a) median masih dijalankan
> **setelah** penajaman pada varian "urutan lain", sehingga apa pun yang diperkuat tetap tersapu; dan
> (b) degradasi dominan di sini adalah iluminasi, bukan kabur, sehingga penajaman bukan tahap yang menentukan.
> Bandingkan dengan Bab 11, di mana kabur adalah satu-satunya degradasi dan penajaman menurunkan CER dari
> 0,58 menjadi 0,21.
>
> Pelajaran metodologisnya justru yang paling berharga: **aturan urutan pipeline mengodekan mode kegagalan
> yang lazim, bukan jaminan.** Untuk berkas Anda sendiri, ablasi seperti tabel di atas — bukan hafalan urutan —
> yang menentukan pipeline mana yang dipakai.

---
## 17. Ringkasan: Memilih Filter Berdasarkan Degradasi

**Tabel keputusan** — cara membaca: kenali degradasinya dahulu (dari histogram, dari peta simpangan baku lokal,
atau dari asal berkas), baru pilih filternya.

| Degradasi | Tanda diagnostik | Filter yang tepat | Hindari |
|---|---|---|---|
| Derau Gaussian/elektronik | histogram melebar merata | Gaussian $\sigma$ kecil, NLM, bilateral | median besar |
| *Salt & pepper* | puncak terisolasi di 0 & 255 | **median 3×3**, median adaptif | rerata, Gaussian, *midpoint* |
| Hanya *salt* (bintik putih) | puncak di 255 saja | filter **minimum**, kontraharmonik $Q<0$ | filter maksimum |
| Hanya *pepper* (bintik hitam) | puncak di 0 saja | filter **maksimum**, kontraharmonik $Q>0$ | filter minimum |
| Impuls kerapatan tinggi (>20%) | citra penuh bintik | **median adaptif** | median tetap berukuran besar |
| Derau campuran | ekor + sebaran melebar | **alpha-trimmed** $d$ menengah | filter tunggal apa pun |
| Derau seragam/kuantisasi | histogram bertangga | *midpoint*, Gaussian | median (tidak menekan derau seragam) |
| Kabur/defokus | tepi melandai | **unsharp masking**, high-boost | Laplacian pada citra berderau |
| Kontras rendah/kusam | rentang histogram sempit | Laplacian, unsharp, CLAHE | penajaman berlebih ($k$ besar) |
| Iluminasi tidak rata | rerata lokal bervariasi | koreksi latar (kernel besar), **ambang adaptif**, CLAHE | Otsu global |
| Perlu peta tepi/fitur | — | **Sobel**/Scharr (setelah denoise) | Roberts pada citra berderau |
| Perlu halus tapi tepi tajam | area foto di dokumen | **bilateral**, NLM | Gaussian, rerata |
| Dokumen miring | garis kop tidak horizontal | Sobel $g_y$ + Hough | memaksa OCR pada citra miring |

**Empat kaidah yang layak dihafal:**

1. **Linear untuk derau aditif, statistik urutan untuk derau impuls.** Memakai rerata pada *salt & pepper*
   memperburuk keadaan, bukan sedikit, tetapi jauh.
2. **Denoise sebelum menajamkan; binarisasi paling akhir.**
3. **Ukuran kernel adalah pernyataan tentang skala objek.** Kernel harus lebih kecil dari detail yang ingin
   dipertahankan, dan lebih besar dari derau yang ingin dibuang. Bila kedua syarat itu berbenturan, jawabannya
   adalah filter adaptif, bukan kernel kompromi.
4. **Evaluasi dengan metrik tugas hilir.** PSNR/SSIM berguna untuk penyetelan, tetapi keputusan akhir mengikuti
   ukuran yang dipakai sistem sebenarnya (CER OCR, keberhasilan deteksi tanda tangan).

In [ ]:
# Rekapitulasi eksperimental: satu tabel akhir untuk seluruh notebook
kasus = {
    "Gaussian s=35":  derau_gaussian(IJAZAH, 35, seed=7),
    "impuls 8%":      derau_impulse(IJAZAH, 0.08, seed=1),
    "impuls 35%":     derau_impulse(IJAZAH, 0.35, seed=13),
    "campuran":       derau_impulse(derau_seragam(IJAZAH, 30, seed=2), 0.04, seed=3),
    "defokus s=2.2":  kabur_defokus(IJAZAH, 2.2),
}
filter_uji = {
    "tanpa filter":      lambda x: x,
    "rerata 3x3":        lambda x: cv2.blur(x, (3, 3)),
    "Gaussian s=1.0":    lambda x: cv2.GaussianBlur(x, (0, 0), 1.0),
    "median 3x3":        lambda x: cv2.medianBlur(x, 3),
    "median adaptif":    lambda x: median_adaptif(x, 7),
    "alpha-trim d=4":    lambda x: rerata_terpangkas_alfa(x, 3, 4),
    "bilateral":         lambda x: cv2.bilateralFilter(x, 9, 75, 9),
    "unsharp k=1":       lambda x: unsharp_masking(x, 1.2, 1.0)[0],
}

lebar = max(len(n) for n in filter_uji) + 2
print(f"{'Filter':<{lebar}}" + "".join(f"{k:>16}" for k in kasus))
print("-" * (lebar + 16 * len(kasus)))
tabel = {}
for fn, f in filter_uji.items():
    baris = [psnr(IJAZAH, f(img), data_range=255) for img in kasus.values()]
    tabel[fn] = baris
    print(f"{fn:<{lebar}}" + "".join(f"{v:>16.2f}" for v in baris))

print("\nPemenang per kasus (PSNR tertinggi):")
for j, k in enumerate(kasus):
    juara = max(((fn, v[j]) for fn, v in tabel.items()), key=lambda t: t[1])
    print(f"  {k:<16} -> {juara[0]:<18} ({juara[1]:.2f} dB)")
print("\nTidak ada satu filter yang menang di semua kolom. Itulah inti bab ini.")

---
## 18. Latihan Mandiri

Kerjakan pada notebook ini (tambahkan sel baru di bawah). Setiap jawaban harus memuat **kode, angka
(PSNR/SSIM/CER), dan satu paragraf analisis** — bukan tangkapan layar saja.

**Latihan 1 — Implementasi manual (bobot 15).**
Tulis `median_manual(img, k)` tanpa `cv2.medianBlur`, memakai `sliding_window_view`. Verifikasi hasilnya
identik dengan OpenCV (`np.array_equal`) pada `derau_impulse(IJAZAH, 0.1)`, lalu bandingkan waktu eksekusinya.
Jelaskan mengapa OpenCV jauh lebih cepat (petunjuk: histogram bergulir / *running histogram*).

**Latihan 2 — Kernel yang salah arah (bobot 10).**
Terapkan hanya `Sobel gx` (bukan magnitudo) pada `IJAZAH`, lalu jelaskan mengapa garis kop yang horizontal
**hampir hilang** sementara bingkai vertikal justru menonjol. Buktikan dengan menghitung energi gradien pada
ROI `kop` untuk $g_x$ dan $g_y$ secara terpisah.

**Latihan 3 — Pemilihan filter berbasis diagnosis (bobot 20).**
Buat fungsi `diagnosa(img)` yang menebak jenis derau **hanya dari histogram dan statistik lokal** (kembalikan
salah satu dari `"impuls"`, `"gaussian"`, `"campuran"`, `"bersih"`), lalu `perbaiki_otomatis(img)` yang memilih
filter berdasarkan hasil diagnosis. Uji pada lima citra terdegradasi dari Bab 17 dan laporkan berapa yang
terdiagnosis benar.

**Latihan 4 — Filter kontraharmonik adaptif (bobot 20).**
Kontraharmonik memerlukan tanda $Q$ yang benar. Buat versi yang menentukan tanda $Q$ **per piksel** dari
kecondongan (*skewness*) lokal jendela, lalu bandingkan PSNR-nya dengan kontraharmonik $Q$ tetap pada citra yang
terkena *salt* dan *pepper* dengan proporsi tidak seimbang (mis. `rasio_salt=0.8`).

**Latihan 5 — Optimasi pipeline untuk CER (bobot 25).**
Ambil `BERKAS` dari Bab 16. Lakukan pencarian *grid* atas tiga parameter: ukuran median $\{3,5\}$, $\sigma$
unsharp $\{0{,}8;\ 1{,}2;\ 2{,}0\}$, dan $k$ unsharp $\{0{,}5;\ 1{,}0;\ 1{,}5\}$ — total 18 kombinasi.
Laporkan kombinasi dengan CER terendah, sajikan tabelnya, dan jelaskan apakah kombinasi CER-terbaik sama dengan
kombinasi PSNR-terbaik. Bila berbeda, jelaskan penyebabnya.

**Latihan 6 — Refleksi (bobot 10).**
Pada Bab 4, PSNR dan SSIM memberi kesimpulan berbeda tentang ukuran kernel optimum. Jelaskan asal perbedaan itu
dari rumus kedua metrik, dan usulkan satu metrik ketiga yang lebih sesuai untuk citra dokumen beserta alasannya.

---

### Rujukan

- Gonzalez, R. C. & Woods, R. E. *Digital Image Processing*, 4th ed., Pearson, 2018 — Bab 3 (*Intensity
  Transformations and Spatial Filtering*) dan Bab 5 (*Image Restoration*, untuk filter statistik urutan).
- Tomasi, C. & Manduchi, R. "Bilateral Filtering for Gray and Color Images", *ICCV*, 1998.
- Buades, A., Coll, B. & Morel, J.-M. "A Non-Local Algorithm for Image Denoising", *CVPR*, 2005.
- Hwang, H. & Haddad, R. A. "Adaptive Median Filters: New Algorithms and Results", *IEEE TIP* 4(4), 1995.
- Dokumentasi OpenCV: modul `imgproc` — *Image Filtering*.

*Catatan: rujukan di atas ditulis dari ingatan tanpa akses ke basis data pustaka, jadi mohon periksa kembali
detail halaman, volume, dan tahunnya sebelum dicantumkan pada RPS atau bahan ajar resmi.*